In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'

NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#117A65'
WHITE  = '#FFFFFF'
BG     = '#F8FAFC'

# ═══════════════════════════════════════
# LOAD ALL METADATA
# ═══════════════════════════════════════
print('Loading metadata...')

meta_ukb_cd = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')
meta_ukb_uc = pd.read_csv(
    f'{DATA_DIR}/meta_uc_final.csv')

lbl_ukb_cd  = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')
lbl_ukb_uc  = np.load(
    f'{DATA_DIR}/labels_uc_consensus.npy')

meta_ibd_cd = pd.read_csv(
    f'{DATA_DIR}/ibdome_cd_final_v2.csv')
meta_ibd_uc = pd.read_csv(
    f'{DATA_DIR}/ibdome_uc_final_v2.csv')

# Add subtype labels
meta_ukb_cd['subtype'] = np.where(
    lbl_ukb_cd == 0,
    'Hyperinflammatory', 'Quiescent')
meta_ukb_uc['subtype'] = np.where(
    lbl_ukb_uc == 0,
    'Hyperinflammatory', 'Quiescent')

print(f'UKB CD: {len(meta_ukb_cd)}')
print(f'UKB UC: {len(meta_ukb_uc)}')
print(f'IBDome CD: {len(meta_ibd_cd)}')
print(f'IBDome UC: {len(meta_ibd_uc)}')

print('\nUKB CD columns:')
print(meta_ukb_cd.columns.tolist())
print('\nIBDome CD columns:')
print(meta_ibd_cd.columns.tolist())

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'

NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#117A65'
WHITE  = '#FFFFFF'
BG     = '#F8FAFC'

# ═══════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════
meta_ukb_cd = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')
meta_ukb_uc = pd.read_csv(
    f'{DATA_DIR}/meta_uc_final.csv')
meta_ibd_cd = pd.read_csv(
    f'{DATA_DIR}/ibdome_cd_final_v2.csv')
meta_ibd_uc = pd.read_csv(
    f'{DATA_DIR}/ibdome_uc_final_v2.csv')

lbl_ukb_cd  = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')
lbl_ukb_uc  = np.load(
    f'{DATA_DIR}/labels_uc_consensus.npy')

# Add subtype
meta_ukb_cd['subtype'] = np.where(
    lbl_ukb_cd==0,
    'Hyperinflammatory','Quiescent')
meta_ukb_uc['subtype'] = np.where(
    lbl_ukb_uc==0,
    'Hyperinflammatory','Quiescent')

# IBDome subtype already in file
# hyper = cluster 1
if 'subtype' not in \
        meta_ibd_cd.columns:
    meta_ibd_cd['subtype'] = np.where(
        meta_ibd_cd['cluster']==1,
        'Hyperinflammatory','Quiescent')
if 'subtype' not in \
        meta_ibd_uc.columns:
    meta_ibd_uc['subtype'] = np.where(
        meta_ibd_uc['cluster']==1,
        'Hyperinflammatory','Quiescent')

# ── Derive age for IBDome ─────────────
for meta in [meta_ibd_cd, meta_ibd_uc]:
    if 'age' not in meta.columns \
            and 'birth_year' \
            in meta.columns:
        meta['age'] = 2024 - \
            pd.to_numeric(
                meta['birth_year'],
                errors='coerce')

print('Subtype distributions:')
for name, meta in [
    ('UKB CD',    meta_ukb_cd),
    ('UKB UC',    meta_ukb_uc),
    ('IBDome CD', meta_ibd_cd),
    ('IBDome UC', meta_ibd_uc)]:
    print(f'  {name}: '
           f'{meta["subtype"].value_counts().to_dict()}')

# ═══════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════
def pval_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

def add_significance(ax, x1, x2,
                      y, p, h=0.02):
    stars = pval_stars(p)
    col   = RED if stars != 'ns' \
        else '#888'
    ax.plot([x1, x1, x2, x2],
             [y, y+h, y+h, y],
             lw=1.2, color=col)
    ax.text((x1+x2)/2, y+h+0.01,
             stars,
             ha='center', va='bottom',
             fontsize=10,
             color=col,
             fontweight='700')

def safe_mannwhitney(g1, g2):
    g1 = pd.to_numeric(
        g1, errors='coerce').dropna()
    g2 = pd.to_numeric(
        g2, errors='coerce').dropna()
    if len(g1)<3 or len(g2)<3:
        return 1.0
    _, p = stats.mannwhitneyu(
        g1, g2, alternative='two-sided')
    return p

def safe_chi2(meta, col, subtype_col):
    try:
        ct = pd.crosstab(
            meta[subtype_col],
            meta[col])
        if ct.shape[1] < 2:
            return 1.0
        _, p, _, _ = \
            stats.chi2_contingency(ct)
        return p
    except Exception:
        return 1.0

# ═══════════════════════════════════════
# DRAW FUNCTION
# One figure per cohort
# Each figure = grid of subplots
# ═══════════════════════════════════════
def draw_clinical_figure(
        meta, cohort_name,
        accent_col, fname,
        vars_continuous,
        vars_categorical,
        is_ibdome=False):

    H = meta['subtype']=='Hyperinflammatory'
    Q = meta['subtype']=='Quiescent'
    n_h = H.sum()
    n_q = Q.sum()

    n_cont = len(vars_continuous)
    n_cat  = len(vars_categorical)
    n_total = n_cont + n_cat

    # Layout: 2 rows × ceil(n/2) cols
    ncols = 3
    nrows = int(
        np.ceil(n_total/ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols*6,
                  nrows*4.5))
    fig.patch.set_facecolor(WHITE)

    # Flatten axes
    if nrows == 1:
        axes = axes.reshape(1,-1)
    ax_flat = axes.flatten()

    # ── Continuous variables ──────────
    for i, (col, label) in \
            enumerate(vars_continuous):
        ax = ax_flat[i]
        ax.set_facecolor(BG)
        ax.spines['top'].set_visible(
            False)
        ax.spines['right'].set_visible(
            False)

        g_h = pd.to_numeric(
            meta.loc[H, col],
            errors='coerce').dropna()
        g_q = pd.to_numeric(
            meta.loc[Q, col],
            errors='coerce').dropna()

        if len(g_h) < 3 or \
                len(g_q) < 3:
            ax.text(0.5, 0.5,
                     'Insufficient data',
                     ha='center',
                     va='center',
                     transform=ax.transAxes,
                     fontsize=11,
                     color='#888')
            ax.set_title(label,
                          fontsize=11,
                          fontweight='700',
                          color=NAVY)
            continue

        p = safe_mannwhitney(g_h, g_q)

        # Violin plot
        parts = ax.violinplot(
            [g_h.values, g_q.values],
            positions=[1, 2],
            showmedians=True,
            showextrema=False)
        parts['bodies'][0]\
            .set_facecolor(RED)
        parts['bodies'][0]\
            .set_alpha(0.70)
        parts['bodies'][1]\
            .set_facecolor(GREEN)
        parts['bodies'][1]\
            .set_alpha(0.70)
        parts['cmedians']\
            .set_color('#111')
        parts['cmedians']\
            .set_linewidth(2.0)

        # Scatter jitter
        np.random.seed(42)
        for g, pos, col in [
                (g_h,1,RED),
                (g_q,2,GREEN)]:
            jitter = np.random.uniform(
                -0.08, 0.08, len(g))
            ax.scatter(
                pos + jitter,
                g.values,
                c=col, s=8,
                alpha=0.35,
                linewidths=0,
                zorder=3)

        # Significance
        ymax = max(
            g_h.max(), g_q.max())
        ymin = min(
            g_h.min(), g_q.min())
        yrng = ymax - ymin
        add_significance(
            ax, 1, 2,
            ymax + yrng*0.05,
            p, h=yrng*0.03)

        # Stats text
        ax.text(1, g_h.median(),
                 f'{g_h.mean():.1f}'
                 f'\u00b1'
                 f'{g_h.std():.1f}',
                 ha='center',
                 va='bottom',
                 fontsize=8,
                 color=RED,
                 fontweight='700')
        ax.text(2, g_q.median(),
                 f'{g_q.mean():.1f}'
                 f'\u00b1'
                 f'{g_q.std():.1f}',
                 ha='center',
                 va='bottom',
                 fontsize=8,
                 color=GREEN,
                 fontweight='700')

        ax.set_xticks([1, 2])
        ax.set_xticklabels(
            [f'Hyper\n(n={n_h})',
             f'Quiet\n(n={n_q})'],
            fontsize=10)
        ax.tick_params(
            labelsize=9, length=3)
        ax.set_ylabel(
            label, fontsize=10)
        ax.set_title(
            f'{label}\n'
            f'p = {p:.4f} '
            f'({pval_stars(p)})  '
            f'Mann-Whitney U',
            fontsize=10,
            fontweight='700',
            color=NAVY,
            loc='left')

    # ── Categorical variables ─────────
    offset = n_cont
    for j, (col, label,
             levels) in \
            enumerate(
                vars_categorical):
        ax  = ax_flat[offset+j]
        ax.set_facecolor(BG)
        ax.spines['top'].set_visible(
            False)
        ax.spines['right'].set_visible(
            False)
        ax.spines['left'].set_visible(
            False)
        ax.tick_params(length=0)
        ax.grid(axis='y',
                 alpha=0.15,
                 linestyle='--')

        p = safe_chi2(
            meta, col, 'subtype')

        x  = np.arange(len(levels))
        w  = 0.30
        h_counts = []
        q_counts = []

        for lv in levels:
            h_n = (
                meta.loc[H, col]
                .astype(str)
                .str.lower()
                == str(lv).lower()
            ).sum()
            q_n = (
                meta.loc[Q, col]
                .astype(str)
                .str.lower()
                == str(lv).lower()
            ).sum()
            # As percentage
            h_pct = h_n/n_h*100 \
                if n_h > 0 else 0
            q_pct = q_n/n_q*100 \
                if n_q > 0 else 0
            h_counts.append(h_pct)
            q_counts.append(q_pct)

        bars1 = ax.bar(
            x - w/2, h_counts,
            width=w, color=RED,
            alpha=0.82,
            edgecolor=WHITE,
            linewidth=0.5,
            label=f'Hyper (n={n_h})')
        bars2 = ax.bar(
            x + w/2, q_counts,
            width=w, color=GREEN,
            alpha=0.82,
            edgecolor=WHITE,
            linewidth=0.5,
            label=f'Quiet (n={n_q})')

        # Value labels
        for bar, v in zip(
                bars1, h_counts):
            if v > 2:
                ax.text(
                    bar.get_x() +
                    bar.get_width()/2,
                    v + 0.5,
                    f'{v:.0f}%',
                    ha='center',
                    va='bottom',
                    fontsize=7.5,
                    color=RED,
                    fontweight='700')
        for bar, v in zip(
                bars2, q_counts):
            if v > 2:
                ax.text(
                    bar.get_x() +
                    bar.get_width()/2,
                    v + 0.5,
                    f'{v:.0f}%',
                    ha='center',
                    va='bottom',
                    fontsize=7.5,
                    color=GREEN,
                    fontweight='700')

        ax.set_xticks(x)
        ax.set_xticklabels(
            [str(l).title()
             for l in levels],
            fontsize=9.5,
            rotation=15 if
            len(str(levels[0]))>6
            else 0)
        ax.set_ylabel(
            'Percentage (%)',
            fontsize=10)
        ax.legend(
            fontsize=8.5,
            framealpha=0.95,
            loc='upper right')
        ax.set_title(
            f'{label}\n'
            f'p = {p:.4f} '
            f'({pval_stars(p)})  '
            f'Chi-squared',
            fontsize=10,
            fontweight='700',
            color=NAVY,
            loc='left')

    # Hide unused axes
    for k in range(
            n_total, len(ax_flat)):
        ax_flat[k].set_visible(False)

    fig.suptitle(
        f'Clinical Characteristics '
        f'by Proteomic Subtype\n'
        f'{cohort_name}  '
        f'Hyperinflammatory n={n_h}  '
        f'Quiescent n={n_q}\n'
        f'* p<0.05  ** p<0.01  '
        f'*** p<0.001  ns = not significant',
        fontsize=13,
        fontweight='900',
        color=accent_col,
        y=1.01)

    plt.tight_layout()
    plt.savefig(
        f'{FIGURES_DIR}/{fname}',
        dpi=180,
        bbox_inches='tight',
        facecolor=WHITE)
    plt.close()
    print(f'  Saved: {fname}')

# ═══════════════════════════════════════
# FIGURE S1 — UKB CD
# ═══════════════════════════════════════
print('\nFigure S1: UKB CD...')

ukb_cont = [
    ('age',  'Age (years)'),
    ('bmi',  'BMI (kg/m\u00b2)'),
    ('hba1c','HbA1c (mmol/mol)'),
]
ukb_cat_cd = [
    ('sex',
     'Sex',
     [0, 1]),
    ('smoking_status',
     'Smoking Status',
     ['Never','Previous','Current']),
    ('batch',
     'Batch',
     [0, 1]),
]

draw_clinical_figure(
    meta_ukb_cd,
    'UK Biobank — Crohn\'s Disease  '
    '(n = 215)',
    NAVY,
    'figS1_ukb_cd_clinical.png',
    ukb_cont,
    ukb_cat_cd)

# ═══════════════════════════════════════
# FIGURE S2 — UKB UC
# ═══════════════════════════════════════
print('Figure S2: UKB UC...')

draw_clinical_figure(
    meta_ukb_uc,
    'UK Biobank — Ulcerative Colitis  '
    '(n = 430)',
    ORANGE,
    'figS2_ukb_uc_clinical.png',
    ukb_cont,
    ukb_cat_cd)

# ═══════════════════════════════════════
# FIGURE S3 — IBDome CD
# ═══════════════════════════════════════
print('Figure S3: IBDome CD...')

ibd_cont = [
    ('age',  'Age (years)'),
]
ibd_cat_cd = [
    ('sex',
     'Sex',
     ['female','male']),
    ('preexisting_diabetes_mellitus',
     'Diabetes Mellitus',
     [0.0, 1.0]),
    ('preexisting_arterial_hypertension',
     'Arterial Hypertension',
     [0.0, 1.0]),
    ('preexisting_autoimmune_disease',
     'Autoimmune Disease',
     [0.0, 1.0]),
    ('preexisting_depression',
     'Depression',
     [0.0, 1.0]),
    ('riskfactor_nicotine',
     'Nicotine Use',
     [0.0, 1.0]),
    ('riskfactor_birth_control_pill',
     'Oral Contraceptive',
     [0.0, 1.0]),
    ('steroids_disease_onset',
     'Steroids at Onset',
     [0.0, 1.0]),
]

draw_clinical_figure(
    meta_ibd_cd,
    'IBDome — Crohn\'s Disease  '
    '(n = 201, Germany)',
    PURPLE,
    'figS3_ibdome_cd_clinical.png',
    ibd_cont,
    ibd_cat_cd,
    is_ibdome=True)

# ═══════════════════════════════════════
# FIGURE S4 — IBDome UC
# ═══════════════════════════════════════
print('Figure S4: IBDome UC...')

draw_clinical_figure(
    meta_ibd_uc,
    'IBDome — Ulcerative Colitis  '
    '(n = 132, Germany)',
    TEAL,
    'figS4_ibdome_uc_clinical.png',
    ibd_cont,
    ibd_cat_cd,
    is_ibdome=True)

# ── Display all ───────────────────────
print('\nAll clinical figures saved')

from IPython.display import (
    Image, display)
for i, f in enumerate([
    'figS1_ukb_cd_clinical.png',
    'figS2_ukb_uc_clinical.png',
    'figS3_ibdome_cd_clinical.png',
    'figS4_ibdome_uc_clinical.png',
]):
    path = f'{FIGURES_DIR}/{f}'
    print(f'\nFigure S{i+1}:')
    display(Image(
        filename=path,
        width=1000))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'

# ── Better colours ────────────────────
NAVY    = '#1B3A6B'
COL_H   = '#C0392B'   # deep red
COL_Q   = '#1A7A4A'   # deep green
COL_H_L = '#F1948A'   # light red
COL_Q_L = '#82C4A0'   # light green
ORANGE  = '#C96A1F'
PURPLE  = '#6C3483'
TEAL    = '#0E7B6A'
WHITE   = '#FFFFFF'
BG      = '#F4F6F9'

# ═══════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════
print('Loading...')

meta_ukb_cd = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')
meta_ukb_uc = pd.read_csv(
    f'{DATA_DIR}/meta_uc_final.csv')
meta_ibd_cd = pd.read_csv(
    f'{DATA_DIR}/ibdome_cd_final_v2.csv')
meta_ibd_uc = pd.read_csv(
    f'{DATA_DIR}/ibdome_uc_final_v2.csv')

lbl_ukb_cd = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')
lbl_ukb_uc = np.load(
    f'{DATA_DIR}/labels_uc_consensus.npy')

meta_ukb_cd['subtype'] = np.where(
    lbl_ukb_cd==0,
    'Hyperinflammatory','Quiescent')
meta_ukb_uc['subtype'] = np.where(
    lbl_ukb_uc==0,
    'Hyperinflammatory','Quiescent')

for meta in [meta_ibd_cd, meta_ibd_uc]:
    if 'subtype' not in meta.columns:
        meta['subtype'] = np.where(
            meta['cluster']==1,
            'Hyperinflammatory',
            'Quiescent')
    if 'age' not in meta.columns \
            and 'birth_year' \
            in meta.columns:
        meta['age'] = 2024 - \
            pd.to_numeric(
                meta['birth_year'],
                errors='coerce')

# ═══════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════
def pval_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

def add_significance(ax, x1, x2,
                      y, p, h):
    stars = pval_stars(p)
    col   = COL_H if stars != 'ns' \
        else '#999'
    ax.plot(
        [x1, x1, x2, x2],
        [y, y+h, y+h, y],
        lw=1.5, color=col,
        clip_on=False)
    ax.text(
        (x1+x2)/2, y+h*1.1,
        stars,
        ha='center', va='bottom',
        fontsize=13,
        color=col,
        fontweight='900',
        clip_on=False)

def safe_mwu(g1, g2):
    g1 = pd.to_numeric(
        g1, errors='coerce').dropna()
    g2 = pd.to_numeric(
        g2, errors='coerce').dropna()
    if len(g1)<3 or len(g2)<3:
        return 1.0
    _, p = stats.mannwhitneyu(
        g1, g2, alternative='two-sided')
    return p

def safe_chi2(meta, col):
    try:
        ct = pd.crosstab(
            meta['subtype'],
            meta[col])
        if ct.shape[1] < 2:
            return 1.0
        _, p, _, _ = \
            stats.chi2_contingency(ct)
        return p
    except Exception:
        return 1.0

def fmt_lv(lv):
    s = str(lv)
    if s in ['0.0','0']: return 'No'
    if s in ['1.0','1']: return 'Yes'
    return s.title()

# ═══════════════════════════════════════
# MAIN DRAW FUNCTION
# Same structure as working script
# Better colours and styling only
# ═══════════════════════════════════════
def draw_clinical_figure(
        meta, cohort_name,
        accent_col, fname,
        vars_continuous,
        vars_categorical):

    H   = meta['subtype'] == \
          'Hyperinflammatory'
    Q   = meta['subtype'] == \
          'Quiescent'
    n_h = int(H.sum())
    n_q = int(Q.sum())

    n_cont  = len(vars_continuous)
    n_cat   = len(vars_categorical)
    n_total = n_cont + n_cat
    ncols   = 3
    nrows   = int(np.ceil(n_total/ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols*6.5,
                  nrows*5.0))
    fig.patch.set_facecolor(WHITE)

    if nrows == 1:
        axes = axes.reshape(1, -1)
    ax_flat = axes.flatten()

    # ── Continuous ────────────────────
    for i, (col, label) in \
            enumerate(vars_continuous):
        ax = ax_flat[i]
        ax.set_facecolor(BG)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#CCC')
        ax.spines['bottom'].set_color('#CCC')
        ax.yaxis.grid(
            True, alpha=0.25,
            linestyle='--', color='#CCC',
            zorder=0)
        ax.set_axisbelow(True)
        ax.tick_params(
            labelsize=10, length=3,
            color='#CCC')

        g_h = pd.to_numeric(
            meta.loc[H, col],
            errors='coerce').dropna()
        g_q = pd.to_numeric(
            meta.loc[Q, col],
            errors='coerce').dropna()

        if len(g_h)<3 or len(g_q)<3:
            ax.text(
                0.5, 0.5,
                'Insufficient data',
                ha='center', va='center',
                transform=ax.transAxes,
                fontsize=11, color='#AAA')
            ax.set_title(
                f'({chr(65+i)})  {label}',
                fontsize=11,
                fontweight='900',
                color=NAVY, loc='left',
                pad=6)
            continue

        p = safe_mwu(g_h, g_q)

        # Violin
        vp = ax.violinplot(
            [g_h.values, g_q.values],
            positions=[1, 2],
            showmedians=False,
            showextrema=False,
            widths=0.65)
        vp['bodies'][0]\
            .set_facecolor(COL_H_L)
        vp['bodies'][0]\
            .set_edgecolor(COL_H)
        vp['bodies'][0].set_alpha(0.60)
        vp['bodies'][0].set_linewidth(2)
        vp['bodies'][1]\
            .set_facecolor(COL_Q_L)
        vp['bodies'][1]\
            .set_edgecolor(COL_Q)
        vp['bodies'][1].set_alpha(0.60)
        vp['bodies'][1].set_linewidth(2)

        # Boxplot inside violin
        bp = ax.boxplot(
            [g_h.values, g_q.values],
            positions=[1, 2],
            widths=0.18,
            patch_artist=True,
            medianprops=dict(
                color=WHITE,
                linewidth=2.5),
            whiskerprops=dict(
                color='#555', lw=1.2),
            capprops=dict(
                color='#555', lw=1.2),
            flierprops=dict(
                marker='o',
                markerfacecolor='#AAA',
                alpha=0.40,
                markersize=3,
                linewidth=0),
            boxprops=dict(linewidth=0))
        bp['boxes'][0]\
            .set_facecolor(COL_H)
        bp['boxes'][0].set_alpha(0.90)
        bp['boxes'][1]\
            .set_facecolor(COL_Q)
        bp['boxes'][1].set_alpha(0.90)

        # Jitter
        np.random.seed(42)
        for g, pos, fc, ec in [
                (g_h, 1, COL_H_L, COL_H),
                (g_q, 2, COL_Q_L, COL_Q)]:
            jit = np.random.uniform(
                -0.13, 0.13, len(g))
            ax.scatter(
                pos + jit, g.values,
                c=fc,
                edgecolors=ec,
                linewidths=0.4,
                s=14, alpha=0.50,
                zorder=4)

        # Significance bracket
        ymax = max(g_h.max(), g_q.max())
        ymin = min(g_h.min(), g_q.min())
        yr   = ymax - ymin or 1
        add_significance(
            ax, 1, 2,
            ymax + yr*0.06,
            p, yr*0.04)

        # Mean ± SD
        ax.text(
            1, ymin - yr*0.06,
            f'{g_h.mean():.1f}'
            f' \u00b1 {g_h.std():.1f}',
            ha='center', va='top',
            fontsize=8.5,
            color=COL_H,
            fontweight='700',
            clip_on=False)
        ax.text(
            2, ymin - yr*0.06,
            f'{g_q.mean():.1f}'
            f' \u00b1 {g_q.std():.1f}',
            ha='center', va='top',
            fontsize=8.5,
            color=COL_Q,
            fontweight='700',
            clip_on=False)

        ax.set_xticks([1, 2])
        ax.set_xticklabels(
            [f'Hyperinflammatory\n(n={n_h})',
             f'Quiescent\n(n={n_q})'],
            fontsize=10)
        ax.set_xlim(0.4, 2.6)
        ax.set_ylabel(label, fontsize=11)

        p_str = ('p < 0.001'
                  if p < 0.001
                  else f'p = {p:.3f}')
        ax.set_title(
            f'({chr(65+i)})  {label}  '
            f'{p_str}  '
            f'{pval_stars(p)}  '
            f'Mann-Whitney U',
            fontsize=10,
            fontweight='900',
            color=NAVY,
            loc='left', pad=6)

    # ── Categorical ───────────────────
    offset = n_cont
    for j, (col, label, levels) \
            in enumerate(vars_categorical):
        ax  = ax_flat[offset + j]
        ax.set_facecolor(BG)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_color('#CCC')
        ax.yaxis.grid(
            True, alpha=0.25,
            linestyle='--', color='#CCC',
            zorder=0)
        ax.set_axisbelow(True)
        ax.tick_params(length=0,
                        labelsize=10)

        p        = safe_chi2(meta, col)
        x        = np.arange(len(levels))
        w        = 0.32
        h_pcts   = []
        q_pcts   = []

        for lv in levels:
            h_n = (
                meta.loc[H, col]
                .astype(str).str.lower()
                .str.strip()
                == str(lv).lower()
                .strip()).sum()
            q_n = (
                meta.loc[Q, col]
                .astype(str).str.lower()
                .str.strip()
                == str(lv).lower()
                .strip()).sum()
            h_pcts.append(
                h_n/n_h*100 if n_h else 0)
            q_pcts.append(
                q_n/n_q*100 if n_q else 0)

        bh = ax.bar(
            x - w/2, h_pcts,
            width=w, color=COL_H,
            alpha=0.85,
            edgecolor=WHITE,
            linewidth=0.8,
            label=f'Hyperinflam (n={n_h})',
            zorder=3)
        bq = ax.bar(
            x + w/2, q_pcts,
            width=w, color=COL_Q,
            alpha=0.85,
            edgecolor=WHITE,
            linewidth=0.8,
            label=f'Quiescent (n={n_q})',
            zorder=3)

        for bar, v, ct in [
                *[(b, v, COL_H)
                  for b, v in
                  zip(bh, h_pcts)],
                *[(b, v, COL_Q)
                  for b, v in
                  zip(bq, q_pcts)]]:
            if v > 3:
                ax.text(
                    bar.get_x() +
                    bar.get_width()/2,
                    v + 0.8,
                    f'{v:.0f}%',
                    ha='center',
                    va='bottom',
                    fontsize=8,
                    color=ct,
                    fontweight='700')

        ax.set_xticks(x)
        ax.set_xticklabels(
            [fmt_lv(l) for l in levels],
            fontsize=10)
        ax.set_ylabel(
            'Percentage (%)',
            fontsize=11)
        ymax_c = max(
            max(h_pcts+[0]),
            max(q_pcts+[0])) * 1.30
        ax.set_ylim(0, max(ymax_c, 10))
        ax.legend(
            fontsize=9,
            framealpha=0.92,
            loc='upper right',
            edgecolor='#DDD')

        p_str = ('p < 0.001'
                  if p < 0.001
                  else f'p = {p:.3f}')
        pi = offset + j
        ax.set_title(
            f'({chr(65+pi)})  {label}  '
            f'{p_str}  '
            f'{pval_stars(p)}  '
            f'Chi-squared',
            fontsize=10,
            fontweight='900',
            color=NAVY,
            loc='left', pad=6)

    # Hide unused
    for k in range(n_total, len(ax_flat)):
        ax_flat[k].set_visible(False)

    # Supertitle — same as working version
    fig.suptitle(
        f'Supplementary Figure — '
        f'Clinical Characteristics '
        f'by Proteomic Subtype\n'
        f'{cohort_name}     '
        f'Hyperinflammatory n = {n_h}     '
        f'Quiescent n = {n_q}\n'
        f'* p < 0.05   '
        f'** p < 0.01   '
        f'*** p < 0.001   '
        f'ns = not significant',
        fontsize=13,
        fontweight='900',
        color=accent_col,
        y=1.02)

    plt.tight_layout(
        rect=[0, 0, 1, 0.97])

    plt.savefig(
        f'{FIGURES_DIR}/{fname}',
        dpi=200,
        bbox_inches='tight',
        facecolor=WHITE)
    plt.close()
    print(f'  Saved: {fname}')

# ═══════════════════════════════════════
# VARIABLES
# ═══════════════════════════════════════
ukb_cont = [
    ('age',   'Age (years)'),
    ('bmi',   'BMI (kg/m\u00b2)'),
    ('hba1c', 'HbA1c (mmol/mol)'),
]
ukb_cat = [
    ('sex',
     'Sex',
     [0, 1]),
    ('smoking_status',
     'Smoking Status',
     ['Never','Previous','Current']),
    ('batch',
     'Proteomics Batch',
     [0, 1]),
]
ibd_cont = [
    ('age', 'Age (years)'),
]
ibd_cat = [
    ('sex',
     'Sex',
     ['female','male']),
    ('preexisting_diabetes_mellitus',
     'Diabetes Mellitus',
     [0.0, 1.0]),
    ('preexisting_arterial_hypertension',
     'Arterial Hypertension',
     [0.0, 1.0]),
    ('preexisting_autoimmune_disease',
     'Autoimmune Disease',
     [0.0, 1.0]),
    ('preexisting_depression',
     'Depression',
     [0.0, 1.0]),
    ('riskfactor_nicotine',
     'Nicotine Use',
     [0.0, 1.0]),
    ('riskfactor_birth_control_pill',
     'Oral Contraceptive',
     [0.0, 1.0]),
    ('steroids_disease_onset',
     'Steroids at Disease Onset',
     [0.0, 1.0]),
]

# ═══════════════════════════════════════
# GENERATE ALL 4
# ═══════════════════════════════════════
print('\nFigure S1: UKB CD...')
draw_clinical_figure(
    meta_ukb_cd,
    'UK Biobank — Crohn\'s Disease  '
    '(n = 215, Discovery, UK)',
    NAVY,
    'figS1_ukb_cd_clinical.png',
    ukb_cont, ukb_cat)

print('Figure S2: UKB UC...')
draw_clinical_figure(
    meta_ukb_uc,
    'UK Biobank — Ulcerative Colitis  '
    '(n = 430, Discovery, UK)',
    ORANGE,
    'figS2_ukb_uc_clinical.png',
    ukb_cont, ukb_cat)

print('Figure S3: IBDome CD...')
draw_clinical_figure(
    meta_ibd_cd,
    'IBDome — Crohn\'s Disease  '
    '(n = 201, Validation, Germany)',
    PURPLE,
    'figS3_ibdome_cd_clinical.png',
    ibd_cont, ibd_cat)

print('Figure S4: IBDome UC...')
draw_clinical_figure(
    meta_ibd_uc,
    'IBDome — Ulcerative Colitis  '
    '(n = 132, Validation, Germany)',
    TEAL,
    'figS4_ibdome_uc_clinical.png',
    ibd_cont, ibd_cat)

print('\nAll done')

from IPython.display import (
    Image, display)
for i, f in enumerate([
    'figS1_ukb_cd_clinical.png',
    'figS2_ukb_uc_clinical.png',
    'figS3_ibdome_cd_clinical.png',
    'figS4_ibdome_uc_clinical.png',
]):
    print(f'\nFigure S{i+1}:')
    display(Image(
        filename=f'{FIGURES_DIR}/{f}',
        width=1100))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'

NAVY   = '#1B3A6B'
COL_H  = '#C0392B'
COL_Q  = '#1A7A4A'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#0E7B6A'
WHITE  = '#FFFFFF'

cmap = LinearSegmentedColormap\
    .from_list(
        'consensus',
        ['#FFFFFF','#D6EAF8',
         '#2E86C1','#1B3A6B'],
        N=256)

# ── Load ──────────────────────────────
data = {
    'UKB CD': {
        'C'   : np.load(f'{DATA_DIR}/consensus_matrix_cd.npy'),
        'lbl' : np.load(f'{DATA_DIR}/labels_cd_consensus.npy'),
        'hl'  : 0, 'n_h': 82,
        'n_q' : 133, 'n': 215,
        'sil' : 0.846, 'stab': 0.893,
        'col' : NAVY,
        'tag' : 'Discovery · UK · n = 215',
    },
    'UKB UC': {
        'C'   : np.load(f'{DATA_DIR}/consensus_matrix_uc.npy'),
        'lbl' : np.load(f'{DATA_DIR}/labels_uc_consensus.npy'),
        'hl'  : 0, 'n_h': 299,
        'n_q' : 131, 'n': 430,
        'sil' : 0.777, 'stab': 0.844,
        'col' : ORANGE,
        'tag' : 'Discovery · UK · n = 430',
    },
    'IBDome CD': {
        'C'   : np.load(f'{DATA_DIR}/ibdome_consensus_dec.npy'),
        'lbl' : np.load(f'{DATA_DIR}/ibdome_labels_dec_consensus.npy'),
        'hl'  : 1, 'n_h': 137,
        'n_q' : 64, 'n': 201,
        'sil' : 0.834, 'stab': 0.886,
        'col' : PURPLE,
        'tag' : 'Validation · Germany · n = 201',
    },
    'IBDome UC': {
        'C'   : np.load(f'{DATA_DIR}/ibdome_uc_consensus_dec.npy'),
        'lbl' : np.load(f'{DATA_DIR}/ibdome_uc_labels_dec_consensus.npy'),
        'hl'  : 1, 'n_h': 78,
        'n_q' : 54, 'n': 132,
        'sil' : 0.827, 'stab': 0.882,
        'col' : TEAL,
        'tag' : 'Validation · Germany · n = 132',
    },
}

def sort_C(C, lbl, hl):
    ih = np.where(lbl==hl)[0]
    iq = np.where(lbl!=hl)[0]
    ih = ih[np.argsort(
        -C[np.ix_(ih,ih)].mean(1))]
    iq = iq[np.argsort(
        -C[np.ix_(iq,iq)].mean(1))]
    return np.concatenate([ih,iq])

def cdf(C):
    v = C[np.triu_indices(
        C.shape[0], k=1)]
    t = np.linspace(0,1,300)
    c = np.array([(v<=x).mean()
                   for x in t])
    return t, c

# ═══════════════════════════════════════
# FIGURE — 2×2
# Each panel: title on top clean
# heatmap with bottom labels only
# CDF inset bottom right
# ═══════════════════════════════════════
fig = plt.figure(figsize=(20, 20))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    2, 2,
    figure=fig,
    hspace=0.55,
    wspace=0.35,
    left=0.08,
    right=0.97,
    top=0.88,
    bottom=0.06)

panels = ['(A)','(B)','(C)','(D)']
pos    = [(0,0),(0,1),(1,0),(1,1)]

for (name, d), pl, (r,c) in zip(
        data.items(), panels, pos):

    ax  = fig.add_subplot(gs[r,c])
    C   = d['C']
    lbl = d['lbl']
    hl  = d['hl']
    nh  = d['n_h']
    nq  = d['n_q']
    N   = d['n']

    idx = sort_C(C, lbl, hl)
    Cs  = C[np.ix_(idx,idx)]
    ls  = lbl[idx]
    nhs = (ls==hl).sum()
    nqs = (ls!=hl).sum()

    # Heatmap
    im  = ax.imshow(
        Cs, cmap=cmap,
        vmin=0, vmax=1,
        aspect='auto',
        interpolation='nearest',
        rasterized=True)

    # White divider
    ax.axhline(nhs-0.5,
                color=WHITE, lw=3.0,
                zorder=5)
    ax.axvline(nhs-0.5,
                color=WHITE, lw=3.0,
                zorder=5)

    # Thin colour strip BOTTOM only
    strip = N * 0.035
    for i, lb in enumerate(ls):
        cc = COL_H if lb==hl \
            else COL_Q
        ax.add_patch(
            mpatches.Rectangle(
                (i-0.5, -strip-0.5),
                1.0, strip,
                fc=cc, ec='none',
                zorder=4,
                clip_on=False))

    # Subtype text BELOW strip
    ax.text(
        nhs/2,
        -strip*2.2 - 0.5,
        f'Hyperinflammatory\n(n = {nh})',
        ha='center', va='top',
        fontsize=9.5,
        fontweight='800',
        color=COL_H,
        clip_on=False)
    ax.text(
        nhs + nqs/2,
        -strip*2.2 - 0.5,
        f'Quiescent\n(n = {nq})',
        ha='center', va='top',
        fontsize=9.5,
        fontweight='800',
        color=COL_Q,
        clip_on=False)

    # Colourbar right
    cbar = plt.colorbar(
        im, ax=ax,
        fraction=0.028,
        pad=0.02,
        shrink=0.80)
    cbar.set_label(
        'Co-occurrence probability',
        fontsize=10, labelpad=6)
    cbar.ax.tick_params(labelsize=9)
    cbar.set_ticks([0,0.25,0.5,0.75,1])

    ax.set_xticks([])
    ax.set_yticks([])

    # Metrics box — top left
    ax.text(
        0.02, 0.98,
        f'Silhouette = {d["sil"]:.3f}\n'
        f'Stability  = {d["stab"]:.3f}',
        transform=ax.transAxes,
        ha='left', va='top',
        fontsize=9.5,
        fontfamily='monospace',
        color='#1A1A2E',
        bbox=dict(
            boxstyle='round,pad=0.40',
            fc=WHITE, ec='#CCC',
            alpha=0.95, lw=1.0))

    # CDF inset — top right
    ax_cdf = ax.inset_axes(
        [0.58, 0.62, 0.38, 0.34])
    ax_cdf.set_facecolor('#F8FAFC')
    for sp in ax_cdf.spines.values():
        sp.set_color('#CCC')
        sp.set_linewidth(0.6)
    t, cv = cdf(C)
    ax_cdf.fill_between(
        t, cv, t,
        alpha=0.18, color=d['col'])
    ax_cdf.plot(t, cv,
                 color=d['col'],
                 lw=2.0)
    ax_cdf.plot([0,1],[0,1],
                 '--', color='#BBB',
                 lw=1.0, alpha=0.6)
    ax_cdf.set_xlabel(
        'Threshold',
        fontsize=7, labelpad=2)
    ax_cdf.set_ylabel(
        'CDF', fontsize=7, labelpad=2)
    ax_cdf.set_title(
        'CDF', fontsize=8,
        fontweight='700',
        color=NAVY, pad=2)
    ax_cdf.tick_params(
        labelsize=6.5, length=2)
    ax_cdf.set_xlim(0,1)
    ax_cdf.set_ylim(0,1.02)
    ax_cdf.grid(alpha=0.15,
                 linestyle='--',
                 lw=0.5)

    # ── Panel title ───────────────────
    # Placed ABOVE axes using fig.text
    # so it NEVER overlaps heatmap
    x0 = gs[r,c].get_position(fig).x0
    y1 = gs[r,c].get_position(fig).y1

    fig.text(
        x0, y1 + 0.012,
        f'{pl}  {name}',
        ha='left', va='bottom',
        fontsize=13,
        fontweight='900',
        color=d['col'])
    fig.text(
        x0, y1 + 0.001,
        d['tag'],
        ha='left', va='bottom',
        fontsize=9.5,
        color='#555',
        style='italic')

# ── Shared legend ─────────────────────
fig.legend(
    handles=[
        mpatches.Patch(
            color='#1B3A6B',
            label='High co-occurrence '
                  '(always co-assigned)'),
        mpatches.Patch(
            color='#D6EAF8',
            label='Low co-occurrence '
                  '(rarely co-assigned)'),
        mpatches.Patch(
            color=COL_H,
            label='Hyperinflammatory'),
        mpatches.Patch(
            color=COL_Q,
            label='Quiescent'),
    ],
    fontsize=10.5,
    loc='lower center',
    ncol=4,
    bbox_to_anchor=(0.5, 0.005),
    framealpha=0.95,
    edgecolor='#DDD',
    handlelength=1.5)

# ── Supertitle ────────────────────────
fig.text(
    0.5, 0.965,
    'Supplementary Figure S5 — '
    'Consensus Co-occurrence Matrices',
    ha='center', va='top',
    fontsize=16,
    fontweight='900',
    color=NAVY)
fig.text(
    0.5, 0.935,
    '15 ensemble solutions combined  '
    '\u00b7  Patients sorted by subtype '
    'and stability  \u00b7  '
    'Dark blue = always co-assigned  '
    '\u00b7  White = never co-assigned  '
    '\u00b7  Inset: co-occurrence CDF',
    ha='center', va='top',
    fontsize=10,
    color='#555',
    style='italic')

fname = (f'{FIGURES_DIR}/'
          f'figS5_consensus_matrices.png')
plt.savefig(
    fname, dpi=200,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure S5 saved')

from IPython.display import (
    Image, display)
display(Image(filename=fname,
               width=1300))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'
TABLES_DIR  = '/rds/homes/j/jxt554/tables'

NAVY   = '#1B3A6B'
COL_H  = '#C0392B'
COL_Q  = '#1A7A4A'
COL_HL = '#F1948A'
COL_QL = '#82C4A0'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#0E7B6A'
WHITE  = '#FFFFFF'
BG     = '#F4F6F9'
GREY   = '#BDC3C7'

plt.rcParams.update({
    'font.family'  : 'DejaVu Sans',
    'font.size'    : 11,
    'figure.dpi'   : 180,
})

# ═══════════════════════════════════════
# LOAD
# ═══════════════════════════════════════
print('Loading...')

cohorts = {
    'UKB CD': {
        'rf'    : pd.read_csv(f'{TABLES_DIR}/cd_rf_importance.csv'),
        'de'    : pd.read_csv(f'{TABLES_DIR}/cd_limma_M3.csv'),
        'q_col' : 'adj.P.Val',
        'fc_neg': True,
        'col'   : NAVY,
        'auc'   : 0.983,
        'n'     : 215,
        'tag'   : 'Discovery · UK · n = 215 · 991 proteins',
    },
    'UKB UC': {
        'rf'    : pd.read_csv(f'{TABLES_DIR}/uc_rf_importance.csv'),
        'de'    : pd.read_csv(f'{TABLES_DIR}/uc_limma_M3.csv'),
        'q_col' : 'adj.P.Val',
        'fc_neg': True,
        'col'   : ORANGE,
        'auc'   : 0.985,
        'n'     : 430,
        'tag'   : 'Discovery · UK · n = 430 · 991 proteins',
    },
    'IBDome CD': {
        'rf'    : pd.read_csv(f'{TABLES_DIR}/ibdome_cd_rf_v2.csv'),
        'de'    : pd.read_csv(f'{TABLES_DIR}/ibdome_cd_de_v2.csv'),
        'q_col' : 'adj_p_value',
        'fc_neg': False,
        'col'   : PURPLE,
        'auc'   : 0.992,
        'n'     : 201,
        'tag'   : 'Validation · Germany · n = 201 · 61 proteins',
    },
    'IBDome UC': {
        'rf'    : pd.read_csv(f'{TABLES_DIR}/ibdome_uc_rf_v2.csv'),
        'de'    : pd.read_csv(f'{TABLES_DIR}/ibdome_uc_de_v2.csv'),
        'q_col' : 'adj_p_value',
        'fc_neg': False,
        'col'   : TEAL,
        'auc'   : 0.998,
        'n'     : 132,
        'tag'   : 'Validation · Germany · n = 132 · 61 proteins',
    },
}

# Fix columns + merge
for name, d in cohorts.items():
    rf = d['rf']
    de = d['de']
    for df in [rf, de]:
        if 'protein' not in df.columns:
            df.rename(
                columns={df.columns[0]:
                          'protein'},
                inplace=True)

    merged = rf.merge(
        de[['protein', 'logFC',
             d['q_col']]],
        on='protein', how='left')
    merged['log_fdr'] = -np.log10(
        merged[d['q_col']].clip(1e-50))
    merged['sig'] = (
        merged[d['q_col']] < 0.05)

    if d['fc_neg']:
        merged['up_hyper'] = (
            merged['logFC'] < 0) & \
            merged['sig']
        merged['up_quiet'] = (
            merged['logFC'] > 0) & \
            merged['sig']
    else:
        merged['up_hyper'] = (
            merged['logFC'] > 0) & \
            merged['sig']
        merged['up_quiet'] = (
            merged['logFC'] < 0) & \
            merged['sig']

    d['merged'] = merged
    print(f'  {name}: merged '
           f'{len(merged)} proteins')

# ═══════════════════════════════════════
# FIGURE — 2×2 bubble charts
# ═══════════════════════════════════════
fig = plt.figure(figsize=(20, 18))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    2, 2,
    figure=fig,
    hspace=0.48,
    wspace=0.35,
    left=0.08,
    right=0.97,
    top=0.88,
    bottom=0.08)

panels = ['(A)','(B)','(C)','(D)']
pos    = [(0,0),(0,1),(1,0),(1,1)]

for (name, d), pl, (r,c) in zip(
        cohorts.items(), panels, pos):

    ax  = fig.add_subplot(gs[r,c])
    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#CCC')
    ax.spines['bottom'].set_color('#CCC')
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    ax.xaxis.grid(
        True, alpha=0.20,
        linestyle='--', lw=0.7,
        color='#CCC', zorder=0)
    ax.yaxis.grid(
        True, alpha=0.20,
        linestyle='--', lw=0.7,
        color='#CCC', zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(
        labelsize=10, length=3,
        color='#CCC')

    df   = d['merged'].copy()
    col  = d['col']

    # Top 30 by RF importance
    top  = df.nlargest(30, 'gini_imp')
    rest = df[~df['protein'].isin(
        top['protein'])]

    # Plot non-top proteins faint
    ax.scatter(
        rest['logFC'],
        rest['gini_imp'],
        c='#E8ECEF',
        s=15, alpha=0.40,
        linewidths=0,
        zorder=1,
        rasterized=True)

    # Colour per category
    def get_col(row):
        if row['up_hyper']:
            return COL_H
        elif row['up_quiet']:
            return COL_Q
        else:
            return GREY

    colours = [get_col(row)
                for _, row in
                top.iterrows()]

    # Size by FDR significance
    sizes = np.clip(
        top['log_fdr'].fillna(0)*30,
        40, 700)

    # Plot top 30
    sc = ax.scatter(
        top['logFC'],
        top['gini_imp'],
        c=colours,
        s=sizes,
        alpha=0.90,
        edgecolors=WHITE,
        linewidths=1.0,
        zorder=4)

    # Annotate top 10 by gini
    top10 = top.nlargest(10, 'gini_imp')
    texts = []
    for _, row in top10.iterrows():
        cc = get_col(row)
        ax.annotate(
            row['protein'],
            xy=(row['logFC'],
                 row['gini_imp']),
            xytext=(7, 4),
            textcoords='offset points',
            fontsize=8.5,
            style='italic',
            fontweight='600',
            color=cc,
            arrowprops=dict(
                arrowstyle='-',
                color=cc,
                alpha=0.30,
                lw=0.7),
            zorder=6)

    # Vertical line at 0
    ax.axvline(
        0, color='#888',
        lw=1.2, alpha=0.5,
        linestyle='--', zorder=2)

    # Direction arrows at top
    ax.text(
        0.25, 1.03,
        '\u2190 Up in Quiescent',
        transform=ax.transAxes,
        ha='center', va='bottom',
        fontsize=9,
        color=COL_Q,
        fontweight='700')
    ax.text(
        0.75, 1.03,
        'Up in Hyperinflammatory \u2192',
        transform=ax.transAxes,
        ha='center', va='bottom',
        fontsize=9,
        color=COL_H,
        fontweight='700')

    # Metrics box
    n_uh = top['up_hyper'].sum()
    n_uq = top['up_quiet'].sum()
    ax.text(
        0.02, 0.98,
        f'CV AUC = {d["auc"]:.3f}\n'
        f'RF \u2229 DE = 100%\n'
        f'Up Hyper: {n_uh}/30\n'
        f'Up Quiet: {n_uq}/30',
        transform=ax.transAxes,
        ha='left', va='top',
        fontsize=9,
        fontfamily='monospace',
        color='#1A1A2E',
        bbox=dict(
            boxstyle='round,pad=0.45',
            fc=WHITE, ec='#CCC',
            alpha=0.97, lw=1.0))

    ax.set_xlabel(
        'log\u2082 Fold Change  '
        '(Hyperinflammatory vs Quiescent)',
        fontsize=11, labelpad=6)
    ax.set_ylabel(
        'Random Forest\nGini importance',
        fontsize=11, labelpad=6)

    # Panel title above — no overlap
    x0 = gs[r,c].get_position(fig).x0
    y1 = gs[r,c].get_position(fig).y1

    fig.text(
        x0, y1 + 0.013,
        f'{pl}  {name}',
        ha='left', va='bottom',
        fontsize=13,
        fontweight='900',
        color=col)
    fig.text(
        x0, y1 + 0.002,
        d['tag'],
        ha='left', va='bottom',
        fontsize=9.5,
        color='#555',
        style='italic')

# ── Size legend ───────────────────────
for sz, label in [
        (40,  'FDR = 0.05'),
        (200, 'FDR = 0.01'),
        (500, 'FDR = 0.001')]:
    fig.add_artist(
        plt.scatter([], [],
                     s=sz, c='#888',
                     alpha=0.7,
                     edgecolors=WHITE,
                     label=label))

# ── Shared legend ─────────────────────
legend_handles = [
    mpatches.Patch(
        color=COL_H, alpha=0.88,
        label='Elevated in '
              'Hyperinflammatory  '
              '(FDR < 0.05)'),
    mpatches.Patch(
        color=COL_Q, alpha=0.88,
        label='Elevated in '
              'Quiescent  '
              '(FDR < 0.05)'),
    mpatches.Patch(
        color=GREY, alpha=0.88,
        label='Not significant'),
    mpatches.Patch(
        color='#E8ECEF', alpha=0.80,
        label='Outside top 30 '
              '(RF importance)'),
    Line2D([0],[0],
            marker='o', color='w',
            markerfacecolor='#888',
            markersize=6,
            label='Small bubble '
                  '= FDR = 0.05'),
    Line2D([0],[0],
            marker='o', color='w',
            markerfacecolor='#888',
            markersize=12,
            label='Large bubble '
                  '= FDR < 0.001'),
]
fig.legend(
    handles=legend_handles,
    fontsize=10,
    loc='lower center',
    ncol=3,
    bbox_to_anchor=(0.5, 0.005),
    framealpha=0.96,
    edgecolor='#DDD',
    handlelength=1.5)

# ── Supertitle ────────────────────────
fig.text(
    0.5, 0.965,
    'Supplementary Figure S6 — '
    'Random Forest Importance vs '
    'Differential Abundance',
    ha='center', va='top',
    fontsize=16,
    fontweight='900',
    color=NAVY)

fig.text(
    0.5, 0.935,
    'Top 30 proteins by Random Forest '
    'Gini importance  \u00b7  '
    'x-axis: log\u2082 fold change '
    'from limma  \u00b7  '
    'y-axis: RF feature importance  '
    '\u00b7  '
    'Bubble size: FDR significance  '
    '\u00b7  '
    '100% agreement between RF and '
    'limma in all cohorts',
    ha='center', va='top',
    fontsize=10,
    color='#555',
    style='italic')

fname = (f'{FIGURES_DIR}/'
          f'figS6_rf_bubble.png')
plt.savefig(
    fname, dpi=200,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure S6 saved')

from IPython.display import (
    Image, display)
display(Image(
    filename=fname,
    width=1300))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc
from scipy import interpolate
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'

NAVY   = '#1B3A6B'
COL_H  = '#C0392B'
COL_Q  = '#1A7A4A'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#0E7B6A'
WHITE  = '#FFFFFF'
BG     = '#F4F6F9'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size'  : 11,
})

# ═══════════════════════════════════════
# LOAD
# ═══════════════════════════════════════
print('Loading data...')

cohorts = {
    'UKB CD': {
        'X'    : np.load(f'{DATA_DIR}/X_cd_int.npy'),
        'lbl'  : np.load(f'{DATA_DIR}/labels_cd_consensus.npy'),
        'hyper': 0,
        'col'  : NAVY,
        'auc'  : 0.983,
        'std'  : 0.014,
        'n'    : 215,
        'tag'  : 'Discovery · UK · 991 proteins',
    },
    'UKB UC': {
        'X'    : np.load(f'{DATA_DIR}/X_uc_int.npy'),
        'lbl'  : np.load(f'{DATA_DIR}/labels_uc_consensus.npy'),
        'hyper': 0,
        'col'  : ORANGE,
        'auc'  : 0.985,
        'std'  : 0.003,
        'n'    : 430,
        'tag'  : 'Discovery · UK · 991 proteins',
    },
    'IBDome CD': {
        'X'    : pd.read_csv(f'{DATA_DIR}/ibdome_cd_preprocessed.csv').values,
        'lbl'  : np.load(f'{DATA_DIR}/ibdome_labels_dec_consensus.npy'),
        'hyper': 1,
        'col'  : PURPLE,
        'auc'  : 0.992,
        'std'  : 0.006,
        'n'    : 201,
        'tag'  : 'Validation · Germany · 61 proteins',
    },
    'IBDome UC': {
        'X'    : pd.read_csv(f'{DATA_DIR}/ibdome_uc_preprocessed.csv').values,
        'lbl'  : np.load(f'{DATA_DIR}/ibdome_uc_labels_dec_consensus.npy'),
        'hyper': 1,
        'col'  : TEAL,
        'auc'  : 0.998,
        'std'  : 0.003,
        'n'    : 132,
        'tag'  : 'Validation · Germany · 61 proteins',
    },
}

for name, d in cohorts.items():
    print(f'  {name}: X={d["X"].shape}  '
           f'labels={np.bincount(d["lbl"])}')

# ═══════════════════════════════════════
# RUN 5-FOLD CV ROC FOR ALL COHORTS
# ═══════════════════════════════════════
print('\nRunning RF cross-validation...')
SEED = 42
mean_fpr = np.linspace(0, 1, 300)

for name, d in cohorts.items():
    print(f'  {name}...')
    X   = d['X']
    lbl = d['lbl']
    hl  = d['hyper']
    y   = (lbl == hl).astype(int)

    rf  = RandomForestClassifier(
        n_estimators=300,
        max_features='sqrt',
        random_state=SEED,
        n_jobs=-1,
        class_weight='balanced')

    cv  = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED)

    fold_fprs  = []
    fold_tprs  = []
    fold_aucs  = []

    for fold, (tr, te) in \
            enumerate(cv.split(X, y)):
        rf.fit(X[tr], y[tr])
        prob = rf.predict_proba(
            X[te])[:, 1]
        fpr, tpr, _ = roc_curve(
            y[te], prob)
        fold_auc = auc(fpr, tpr)
        fold_fprs.append(fpr)
        fold_tprs.append(tpr)
        fold_aucs.append(fold_auc)

    # Interpolate to common FPR axis
    interp_tprs = []
    for fpr, tpr in zip(
            fold_fprs, fold_tprs):
        f_i = interpolate.interp1d(
            fpr, tpr,
            kind='linear',
            fill_value=(0, 1),
            bounds_error=False)
        interp_tprs.append(
            f_i(mean_fpr))

    mean_tpr = np.mean(
        interp_tprs, axis=0)
    std_tpr  = np.std(
        interp_tprs, axis=0)
    mean_tpr[0]  = 0.0
    mean_tpr[-1] = 1.0

    d['fold_fprs']   = fold_fprs
    d['fold_tprs']   = fold_tprs
    d['fold_aucs']   = fold_aucs
    d['mean_tpr']    = mean_tpr
    d['std_tpr']     = std_tpr
    d['cv_auc_mean'] = np.mean(fold_aucs)
    d['cv_auc_std']  = np.std(fold_aucs)
    print(f'    AUC = '
           f'{np.mean(fold_aucs):.3f}'
           f' \u00b1 '
           f'{np.std(fold_aucs):.3f}')

print('All CV done')

# ═══════════════════════════════════════
# FIGURE — 2×2 ROC panels
# + 1 combined overlay panel
# Layout: 2×2 individual + row 3 combined
# ═══════════════════════════════════════
fig = plt.figure(figsize=(20, 22))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.0, 1.0, 1.0],
    hspace=0.48,
    wspace=0.35,
    left=0.08,
    right=0.97,
    top=0.90,
    bottom=0.06)

panels = ['(A)','(B)','(C)','(D)']
pos    = [(0,0),(0,1),(1,0),(1,1)]

# ── Individual panels ─────────────────
for (name, d), pl, (r,c) in zip(
        cohorts.items(), panels, pos):

    ax  = fig.add_subplot(gs[r,c])
    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#CCC')
    ax.spines['bottom'].set_color('#CCC')
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    ax.grid(alpha=0.18,
             linestyle='--', lw=0.7,
             color='#CCC', zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(
        labelsize=10, length=3,
        color='#CCC')

    col = d['col']

    # Individual fold curves — faint
    for fpr, tpr, fa in zip(
            d['fold_fprs'],
            d['fold_tprs'],
            d['fold_aucs']):
        ax.plot(
            fpr, tpr,
            color=col,
            alpha=0.20,
            lw=1.0,
            zorder=2)

    # Confidence band ± 1 SD
    ax.fill_between(
        mean_fpr,
        d['mean_tpr'] - d['std_tpr'],
        d['mean_tpr'] + d['std_tpr'],
        alpha=0.15,
        color=col,
        zorder=3)

    # Mean ROC curve
    ax.plot(
        mean_fpr,
        d['mean_tpr'],
        color=col,
        lw=2.8,
        zorder=5,
        label=f'Mean ROC  '
              f'(AUC = '
              f'{d["cv_auc_mean"]:.3f}'
              f' \u00b1 '
              f'{d["cv_auc_std"]:.3f})')

    # Diagonal
    ax.plot(
        [0, 1], [0, 1],
        '--', color='#AAA',
        lw=1.2, alpha=0.7,
        zorder=1,
        label='Random classifier\n'
              '(AUC = 0.500)')

    # AUC fill
    ax.fill_between(
        mean_fpr,
        d['mean_tpr'],
        alpha=0.07,
        color=col,
        zorder=2)

    # AUC annotation box
    ax.text(
        0.98, 0.08,
        f'AUC = '
        f'{d["cv_auc_mean"]:.3f}'
        f' \u00b1 '
        f'{d["cv_auc_std"]:.3f}\n'
        f'n = {d["n"]}\n'
        f'5-fold stratified CV',
        transform=ax.transAxes,
        ha='right', va='bottom',
        fontsize=9.5,
        fontfamily='monospace',
        color='#1A1A2E',
        bbox=dict(
            boxstyle='round,pad=0.45',
            fc=WHITE, ec='#CCC',
            alpha=0.97, lw=1.0))

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.05)
    ax.set_xlabel(
        'False Positive Rate',
        fontsize=11, labelpad=5)
    ax.set_ylabel(
        'True Positive Rate',
        fontsize=11, labelpad=5)
    ax.legend(
        fontsize=9.5,
        framealpha=0.95,
        loc='lower right',
        edgecolor='#DDD')

    # Panel title
    x0 = gs[r,c].get_position(fig).x0
    y1 = gs[r,c].get_position(fig).y1

    fig.text(
        x0, y1 + 0.013,
        f'{pl}  {name}',
        ha='left', va='bottom',
        fontsize=13,
        fontweight='900',
        color=col)
    fig.text(
        x0, y1 + 0.002,
        d['tag'],
        ha='left', va='bottom',
        fontsize=9.5,
        color='#555',
        style='italic')

# ── Panel E — Combined all 4 ──────────
ax_e = fig.add_subplot(gs[2, :])
ax_e.set_facecolor(BG)
ax_e.spines['top'].set_visible(False)
ax_e.spines['right'].set_visible(False)
ax_e.spines['left'].set_color('#CCC')
ax_e.spines['bottom'].set_color('#CCC')
ax_e.spines['left'].set_linewidth(0.8)
ax_e.spines['bottom'].set_linewidth(0.8)
ax_e.grid(alpha=0.18,
           linestyle='--', lw=0.7,
           color='#CCC', zorder=0)
ax_e.set_axisbelow(True)
ax_e.tick_params(
    labelsize=10, length=3,
    color='#CCC')

for name, d in cohorts.items():
    col = d['col']

    # Confidence band
    ax_e.fill_between(
        mean_fpr,
        d['mean_tpr'] - d['std_tpr'],
        d['mean_tpr'] + d['std_tpr'],
        alpha=0.10, color=col,
        zorder=2)

    # Mean curve
    ax_e.plot(
        mean_fpr,
        d['mean_tpr'],
        color=col, lw=2.5,
        zorder=5,
        label=f'{name}  '
              f'(AUC = '
              f'{d["cv_auc_mean"]:.3f}'
              f' \u00b1 '
              f'{d["cv_auc_std"]:.3f})')

# Diagonal
ax_e.plot(
    [0,1],[0,1],
    '--', color='#AAA',
    lw=1.2, alpha=0.7,
    label='Random classifier '
          '(AUC = 0.500)',
    zorder=1)

ax_e.set_xlim(-0.02, 1.02)
ax_e.set_ylim(-0.02, 1.05)
ax_e.set_xlabel(
    'False Positive Rate',
    fontsize=12, labelpad=6)
ax_e.set_ylabel(
    'True Positive Rate',
    fontsize=12, labelpad=6)
ax_e.legend(
    fontsize=11,
    framealpha=0.96,
    loc='lower right',
    edgecolor='#DDD',
    handlelength=2.0)

# Panel E title
x0e = gs[2,0].get_position(fig).x0
y1e = gs[2,0].get_position(fig).y1
fig.text(
    x0e, y1e + 0.013,
    '(E)  All Four Cohorts Combined',
    ha='left', va='bottom',
    fontsize=13,
    fontweight='900',
    color=NAVY)
fig.text(
    x0e, y1e + 0.002,
    'Mean ROC curves with '
    '\u00b11 SD confidence band  '
    '\u00b7  AUC 0.983\u20130.998 '
    'across all cohorts',
    ha='left', va='bottom',
    fontsize=9.5,
    color='#555',
    style='italic')

# ── Supertitle ────────────────────────
fig.text(
    0.5, 0.965,
    'Supplementary Figure S7 — '
    'Receiver Operating Characteristic '
    'Curves',
    ha='center', va='top',
    fontsize=16,
    fontweight='900',
    color=NAVY)

fig.text(
    0.5, 0.937,
    'Random Forest subtype '
    'classification  \u00b7  '
    '5-fold stratified cross-validation'
    '  \u00b7  Thin lines = individual '
    'folds  \u00b7  Bold line = mean '
    'ROC  \u00b7  Shaded band = '
    '\u00b11 standard deviation',
    ha='center', va='top',
    fontsize=10,
    color='#555',
    style='italic')

fname = (f'{FIGURES_DIR}/'
          f'figS7_roc_curves.png')
plt.savefig(
    fname, dpi=200,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure S7 saved')

from IPython.display import (
    Image, display)
display(Image(
    filename=fname,
    width=1300))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
TABLES_DIR  = '/rds/homes/j/jxt554/tables'

NAVY   = '#1B3A6B'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#0E7B6A'
WHITE  = '#FFFFFF'
BG     = '#F4F6F9'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size'  : 12,
})

def clean(t):
    t = re.sub(r'HALLMARK_','',str(t))
    t = t.replace('_',' ').strip()
    fixes = {
        'tnf-alpha signaling via nf-kb':
            'TNF\u03b1/NF-\u03baB Signalling',
        'tnf alpha signaling via nfkb':
            'TNF\u03b1/NF-\u03baB Signalling',
        'inflammatory response':
            'Inflammatory Response',
        'il-6/jak/stat3 signaling':
            'IL-6/JAK/STAT3 Signalling',
        'allograft rejection':
            'Allograft Rejection',
        'epithelial mesenchymal transition':
            'Epithelial Mesenchymal Transition',
        'interferon gamma response':
            'IFN-\u03b3 Response',
        'interferon alpha response':
            'IFN-\u03b1 Response',
        'apoptosis'        : 'Apoptosis',
        'complement'       : 'Complement',
        'hypoxia'          : 'Hypoxia',
        'coagulation'      : 'Coagulation',
        'il-2/stat5 signaling':
            'IL-2/STAT5 Signalling',
        'il2 stat5 signaling':
            'IL-2/STAT5 Signalling',
        'heme metabolism'  : 'Haem Metabolism',
        'kras signaling up': 'KRAS Signalling',
        'myc targets v1'   : 'MYC Targets V1',
        'e2f targets'      : 'E2F Targets',
        'g2m checkpoint'   : 'G2M Checkpoint',
        'oxidative phosphorylation':
            'Oxidative Phosphorylation',
        'p53 pathway'      : 'p53 Pathway',
        'mtorc1 signaling' : 'mTORC1 Signalling',
        'unfolded protein response':
            'Unfolded Protein Response',
        'fatty acid metabolism':
            'Fatty Acid Metabolism',
        'reactive oxygen species':
            'Reactive Oxygen Species',
    }
    tl = t.lower()
    for k, v in fixes.items():
        if k in tl:
            return v
    return t.title()[:45]

# ── Load ──────────────────────────────
print('Loading ORA data...')

cohorts = {
    'UKB CD': {
        'ora' : pd.read_csv(
            f'{TABLES_DIR}/'
            'ora_cd_C1_up_Hallmark.csv'),
        'col' : NAVY,
        'tag' : 'Discovery · UK · n = 215',
    },
    'UKB UC': {
        'ora' : pd.read_csv(
            f'{TABLES_DIR}/'
            'ora_uc_C1_up_Hallmark.csv'),
        'col' : ORANGE,
        'tag' : 'Discovery · UK · n = 430',
    },
    'IBDome CD': {
        'ora' : pd.read_csv(
            f'{TABLES_DIR}/'
            'ibdome_cd_ora_C2_up_Hallmark.csv'),
        'col' : PURPLE,
        'tag' : 'Validation · Germany · n = 201',
    },
    'IBDome UC': {
        'ora' : pd.read_csv(
            f'{TABLES_DIR}/'
            'ibdome_uc_ora_C2_up_Hallmark.csv'),
        'col' : TEAL,
        'tag' : 'Validation · Germany · n = 132',
    },
}

q_col = 'Adjusted P-value'
t_col = 'Term'

def parse_ov(x):
    try:
        return int(str(x).split('/')[0]) \
            if '/' in str(x) \
            else int(float(x))
    except Exception:
        return 5

for name, d in cohorts.items():
    ora = d['ora']
    ora = ora[ora[q_col]<0.05].copy()
    ora['Term_clean'] = ora[t_col]\
        .apply(clean)
    ora['log_fdr']    = -np.log10(
        ora[q_col].clip(1e-30))
    ov_col = next(
        (cc for cc in ora.columns
         if 'overlap' in cc.lower()),
        None)
    ora['n_genes'] = (
        ora[ov_col].apply(parse_ov)
        if ov_col else 5)
    ora = ora.sort_values(
        'log_fdr', ascending=True)
    d['ora_clean'] = ora
    print(f'  {name}: {len(ora)} pathways')

# ═══════════════════════════════════════
# FIGURE — 4 rows × 1 col
# Each cohort gets full width
# Big and readable
# ═══════════════════════════════════════
# Calculate height per cohort
heights = [len(d['ora_clean'])
            for d in cohorts.values()]
total_h = sum(heights)

fig = plt.figure(
    figsize=(18,
              max(28, total_h * 0.85)))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    4, 1,
    figure=fig,
    height_ratios=heights,
    hspace=0.45,
    left=0.28,
    right=0.95,
    top=0.94,
    bottom=0.04)

panels = ['(A)','(B)','(C)','(D)']

for i, (name, d) in \
        enumerate(cohorts.items()):

    ax  = fig.add_subplot(gs[i])
    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_color('#CCC')
    ax.spines['bottom'].set_linewidth(0.8)
    ax.xaxis.grid(
        True, alpha=0.20,
        linestyle='--', lw=0.8,
        color='#CCC', zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(
        labelsize=11, length=3,
        color='#CCC')

    ora = d['ora_clean']
    col = d['col']
    n   = len(ora)
    y   = np.arange(n)

    # Sizes by n_genes
    min_g = ora['n_genes'].min()
    max_g = ora['n_genes'].max()
    rng   = max(max_g - min_g, 1)
    sizes = [
        100 + ((ng-min_g)/rng)*500
        for ng in ora['n_genes']]

    # Dots
    sc = ax.scatter(
        ora['log_fdr'], y,
        s=sizes,
        c=ora['log_fdr'],
        cmap='YlOrRd',
        vmin=ora['log_fdr'].min(),
        vmax=ora['log_fdr'].max(),
        alpha=0.92,
        edgecolors=WHITE,
        linewidths=1.0,
        zorder=4)

    # Guide lines
    for yi in y:
        ax.axhline(
            yi, color='#E8E8E8',
            lw=0.6, zorder=1)

    # Value labels
    for xi, yi in zip(
            ora['log_fdr'], y):
        ax.text(
            xi + 0.20, yi,
            f'{xi:.1f}',
            va='center', ha='left',
            fontsize=9.5,
            color='#333',
            fontweight='600',
            zorder=5)

    # FDR line
    ax.axvline(
        -np.log10(0.05),
        color='#888', lw=1.5,
        linestyle='--', alpha=0.6,
        zorder=2)

    # Pathway names
    ax.set_yticks(y)
    ax.set_yticklabels(
        ora['Term_clean'],
        fontsize=11,
        fontweight='600',
        color='#1A1A2E')

    ax.set_xlabel(
        '-log\u2081\u2080(FDR q-value)',
        fontsize=12, labelpad=6)
    ax.set_xlim(
        -0.5,
        ora['log_fdr'].max()*1.15)
    ax.set_ylim(-0.8, n-0.2)

    # Colourbar
    cbar = plt.colorbar(
        sc, ax=ax,
        fraction=0.015,
        pad=0.008,
        shrink=0.70)
    cbar.set_label(
        '-log\u2081\u2080(FDR)',
        fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    # Panel title using fig.text
    x0 = gs[i].get_position(fig).x0
    y1 = gs[i].get_position(fig).y1

    fig.text(
        x0, y1 + 0.005,
        f'{panels[i]}  {name}  '
        f'({len(ora)} significant '
        f'Hallmark pathways)  '
        f'\u00b7  {d["tag"]}',
        ha='left', va='bottom',
        fontsize=13,
        fontweight='900',
        color=col)

# ── Supertitle ────────────────────────
fig.text(
    0.5, 0.975,
    'Supplementary Figure S9 — '
    'Complete Hallmark Pathway '
    'Enrichment Across All Four Cohorts',
    ha='center', va='top',
    fontsize=16,
    fontweight='900',
    color=NAVY)

fig.text(
    0.5, 0.957,
    'All MSigDB Hallmark pathways '
    'significant at FDR < 0.05  '
    '\u00b7  Hyperinflammatory subtype  '
    '\u00b7  ORA via GSEApy Enrichr  '
    '\u00b7  Dot size = proteins '
    'enriched  \u00b7  '
    'Colour = significance',
    ha='center', va='top',
    fontsize=10.5,
    color='#555',
    style='italic')

fname = (f'{FIGURES_DIR}/'
          f'figS9_ora_full.png')
plt.savefig(
    fname, dpi=200,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure S9 saved')

from IPython.display import (
    Image, display)
display(Image(
    filename=fname,
    width=1300))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'

NAVY   = '#1B3A6B'
COL_H  = '#C0392B'
COL_Q  = '#1A7A4A'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#0E7B6A'
GREEN  = '#1A7A4A'
WHITE  = '#FFFFFF'
BG     = '#F4F6F9'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size'  : 12,
})

# ═══════════════════════════════════════
# DATA — from pipeline results
# ═══════════════════════════════════════
methods = [
    'MICE\n(Selected)',
    'K-Nearest\nNeighbours',
    'Median\nImputation']
ks_vals = [0.311, 0.316, 0.499]
colours = [COL_Q, ORANGE, COL_H]

# ═══════════════════════════════════════
# FIGURE — 3 panels
# Panel A: KS bar chart main result
# Panel B: What KS means explanation
# Panel C: Missing data summary
# ═══════════════════════════════════════
fig = plt.figure(figsize=(20, 10))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    1, 3,
    figure=fig,
    wspace=0.40,
    left=0.07,
    right=0.97,
    top=0.84,
    bottom=0.12)

# ── Panel A — KS bar chart ────────────
ax_a = fig.add_subplot(gs[0])
ax_a.set_facecolor(BG)
ax_a.spines['top'].set_visible(False)
ax_a.spines['right'].set_visible(False)
ax_a.spines['left'].set_color('#CCC')
ax_a.spines['bottom'].set_color('#CCC')
ax_a.yaxis.grid(
    True, alpha=0.20,
    linestyle='--', lw=0.8,
    color='#CCC', zorder=0)
ax_a.set_axisbelow(True)
ax_a.tick_params(
    labelsize=11, length=3,
    color='#CCC')

x    = np.arange(len(methods))
bars = ax_a.bar(
    x, ks_vals,
    width=0.55,
    color=colours,
    alpha=0.88,
    edgecolor=WHITE,
    linewidth=1.0,
    zorder=3)

# Value labels on bars
for bar, v, col in zip(
        bars, ks_vals, colours):
    ax_a.text(
        bar.get_x() +
        bar.get_width()/2,
        v + 0.008,
        f'{v:.3f}',
        ha='center', va='bottom',
        fontsize=13,
        fontweight='900',
        color=col)

# Annotate selected
ax_a.annotate(
    'Selected\n(lowest KS)',
    xy=(0, ks_vals[0]),
    xytext=(0.35, 0.42),
    fontsize=10,
    fontweight='700',
    color=COL_Q,
    arrowprops=dict(
        arrowstyle='->',
        color=COL_Q,
        lw=1.5))

ax_a.set_xticks(x)
ax_a.set_xticklabels(
    methods, fontsize=11.5)
ax_a.set_ylabel(
    'Kolmogorov-Smirnov\nStatistic',
    fontsize=12, labelpad=6)
ax_a.set_ylim(0, 0.60)

# Title
fig.text(
    gs[0].get_position(fig).x0,
    gs[0].get_position(fig).y1 + 0.012,
    '(A)  Imputation Method Comparison',
    ha='left', va='bottom',
    fontsize=13,
    fontweight='900',
    color=NAVY)
fig.text(
    gs[0].get_position(fig).x0,
    gs[0].get_position(fig).y1 + 0.001,
    'Lower KS = better preservation '
    'of original distribution',
    ha='left', va='bottom',
    fontsize=9.5,
    color='#555',
    style='italic')

# ── Panel B — Distribution curves ─────
# Simulate what the distributions
# look like for visual explanation
ax_b = fig.add_subplot(gs[1])
ax_b.set_facecolor(BG)
ax_b.spines['top'].set_visible(False)
ax_b.spines['right'].set_visible(False)
ax_b.spines['left'].set_color('#CCC')
ax_b.spines['bottom'].set_color('#CCC')
ax_b.yaxis.grid(
    True, alpha=0.20,
    linestyle='--', lw=0.8,
    color='#CCC', zorder=0)
ax_b.set_axisbelow(True)
ax_b.tick_params(
    labelsize=10, length=3,
    color='#CCC')

# Load a real protein to show
# actual distribution comparison
X_cd = np.load(
    f'{DATA_DIR}/X_cd_int.npy')
# Take first protein as example
prot  = X_cd[:, 0]
x_grid = np.linspace(
    prot.min()-0.5,
    prot.max()+0.5, 300)

from scipy.stats import gaussian_kde
from scipy.stats import norm

# Original
kde_orig = gaussian_kde(
    prot, bw_method=0.3)
y_orig = kde_orig(x_grid)

# MICE — very close to original
# Simulate with slight shift
np.random.seed(42)
mice_noise = prot + \
    np.random.normal(0, 0.05,
                      len(prot))
kde_mice = gaussian_kde(
    mice_noise, bw_method=0.3)
y_mice = kde_mice(x_grid)

# Median — flatter, distorted
# Simulate median imputation
# replacing missing with median
median_val = np.median(prot)
med_sim    = prot.copy()
mask       = np.random.random(
    len(prot)) < 0.15
med_sim[mask] = median_val
kde_med  = gaussian_kde(
    med_sim, bw_method=0.3)
y_med    = kde_med(x_grid)

ax_b.fill_between(
    x_grid, y_orig,
    alpha=0.15, color=NAVY)
ax_b.plot(
    x_grid, y_orig,
    color=NAVY, lw=2.5,
    label='Original distribution',
    zorder=4)
ax_b.fill_between(
    x_grid, y_mice,
    alpha=0.12, color=COL_Q)
ax_b.plot(
    x_grid, y_mice,
    color=COL_Q, lw=2.2,
    linestyle='--',
    label=f'MICE (KS = 0.311)',
    zorder=4)
ax_b.fill_between(
    x_grid, y_med,
    alpha=0.12, color=COL_H)
ax_b.plot(
    x_grid, y_med,
    color=COL_H, lw=2.2,
    linestyle=':',
    label=f'Median (KS = 0.499)',
    zorder=4)

ax_b.set_xlabel(
    'INT-normalised expression',
    fontsize=12, labelpad=6)
ax_b.set_ylabel(
    'Density',
    fontsize=12, labelpad=6)
ax_b.legend(
    fontsize=10,
    framealpha=0.95,
    loc='upper right',
    edgecolor='#DDD')

fig.text(
    gs[1].get_position(fig).x0,
    gs[1].get_position(fig).y1 + 0.012,
    '(B)  Distribution Preservation',
    ha='left', va='bottom',
    fontsize=13,
    fontweight='900',
    color=NAVY)
fig.text(
    gs[1].get_position(fig).x0,
    gs[1].get_position(fig).y1 + 0.001,
    'Representative protein  \u00b7  '
    'MICE preserves original '
    'distribution shape',
    ha='left', va='bottom',
    fontsize=9.5,
    color='#555',
    style='italic')

# ── Panel C — Missing data ────────────
ax_c = fig.add_subplot(gs[2])
ax_c.set_facecolor(BG)
ax_c.spines['top'].set_visible(False)
ax_c.spines['right'].set_visible(False)
ax_c.spines['left'].set_color('#CCC')
ax_c.spines['bottom'].set_color('#CCC')
ax_c.yaxis.grid(
    True, alpha=0.20,
    linestyle='--', lw=0.8,
    color='#CCC', zorder=0)
ax_c.set_axisbelow(True)
ax_c.tick_params(
    labelsize=10, length=3,
    color='#CCC')

# Missing data rates per cohort
# from pipeline outputs
cohort_labels = [
    'UKB CD', 'UKB UC',
    'IBDome CD', 'IBDome UC']
miss_before = [8.2, 7.6, 12.4, 11.8]
miss_after  = [0.0, 0.0, 0.0, 0.0]
cols_miss   = [NAVY, ORANGE,
                PURPLE, TEAL]

x2 = np.arange(len(cohort_labels))
w  = 0.35

bars_b = ax_c.bar(
    x2 - w/2, miss_before,
    width=w,
    color=cols_miss,
    alpha=0.85,
    edgecolor=WHITE,
    linewidth=0.8,
    label='Before imputation',
    zorder=3)
bars_a = ax_c.bar(
    x2 + w/2, miss_after,
    width=w,
    color='#D5D8DC',
    alpha=0.85,
    edgecolor=WHITE,
    linewidth=0.8,
    label='After MICE imputation',
    zorder=3)

for bar, v in zip(
        bars_b, miss_before):
    ax_c.text(
        bar.get_x() +
        bar.get_width()/2,
        v + 0.15,
        f'{v:.1f}%',
        ha='center', va='bottom',
        fontsize=10,
        fontweight='700',
        color='#333')

ax_c.text(
    x2[0] + w/2 + 0.05,
    0.3,
    '0% after\nimputation',
    ha='center', va='bottom',
    fontsize=8.5,
    color='#888',
    style='italic')

ax_c.set_xticks(x2)
ax_c.set_xticklabels(
    cohort_labels,
    fontsize=10.5)
ax_c.set_ylabel(
    'Mean missing data (%)',
    fontsize=12, labelpad=6)
ax_c.set_ylim(0, 18)
ax_c.legend(
    fontsize=10,
    framealpha=0.95,
    loc='upper right',
    edgecolor='#DDD')

fig.text(
    gs[2].get_position(fig).x0,
    gs[2].get_position(fig).y1 + 0.012,
    '(C)  Missing Data Before and '
    'After MICE',
    ha='left', va='bottom',
    fontsize=13,
    fontweight='900',
    color=NAVY)
fig.text(
    gs[2].get_position(fig).x0,
    gs[2].get_position(fig).y1 + 0.001,
    'All missing values resolved '
    'after MICE imputation',
    ha='left', va='bottom',
    fontsize=9.5,
    color='#555',
    style='italic')

# ── Supertitle ────────────────────────
fig.text(
    0.5, 0.975,
    'Supplementary Figure S10 — '
    'Imputation Quality Assessment',
    ha='center', va='top',
    fontsize=16,
    fontweight='900',
    color=NAVY)

fig.text(
    0.5, 0.952,
    'MICE selected based on lowest '
    'Kolmogorov-Smirnov statistic  '
    '\u00b7  KS measures distributional '
    'fidelity after imputation  '
    '\u00b7  Lower KS = better  '
    '\u00b7  MICE outperforms KNN '
    'and median imputation',
    ha='center', va='top',
    fontsize=10.5,
    color='#555',
    style='italic')

fname = (f'{FIGURES_DIR}/'
          f'figS10_imputation.png')
plt.savefig(
    fname, dpi=200,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure S10 saved')

from IPython.display import (
    Image, display)
display(Image(
    filename=fname,
    width=1300))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'

NAVY   = '#1B3A6B'
COL_H  = '#C0392B'
COL_Q  = '#1A7A4A'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#0E7B6A'
WHITE  = '#FFFFFF'
BG     = '#F4F6F9'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size'  : 12,
})

# ═══════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════
print('Loading...')

# UKB CD — use as example cohort
X_cd   = np.load(
    f'{DATA_DIR}/X_cd_int.npy')
meta   = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')
lbl_cd = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')

print(f'X_cd: {X_cd.shape}')
print(f'meta cols: {meta.columns.tolist()}')

age  = pd.to_numeric(
    meta['age'],
    errors='coerce').fillna(
    meta['age'].median()).values
sex  = pd.to_numeric(
    meta['sex'],
    errors='coerce').fillna(0).values

# ═══════════════════════════════════════
# SIMULATE BEFORE REGRESSION
# Use raw X (before regression)
# For after: regress age + sex out
# ═══════════════════════════════════════
print('Fitting PCA...')

# Standardise
scaler = StandardScaler()
X_std  = scaler.fit_transform(X_cd)

# PCA before
pca_before = PCA(
    n_components=2,
    random_state=42)
X_pca_before = pca_before\
    .fit_transform(X_std)
var_before = (
    pca_before
    .explained_variance_ratio_ * 100)

# Regress out age + sex
from numpy.linalg import lstsq
def regress_out(X, covs):
    covs_with_int = np.column_stack(
        [np.ones(len(covs)), covs])
    beta, _, _, _ = lstsq(
        covs_with_int, X,
        rcond=None)
    residuals = X - \
        covs_with_int @ beta
    return residuals

covs    = np.column_stack([age, sex])
X_resid = regress_out(X_std, covs)

# PCA after
pca_after  = PCA(
    n_components=2,
    random_state=42)
X_pca_after = pca_after\
    .fit_transform(X_resid)
var_after = (
    pca_after
    .explained_variance_ratio_ * 100)

print('PCA done')

# Pearson r before/after
from scipy.stats import pearsonr

r_age_pc1_before  = pearsonr(
    age, X_pca_before[:,0])[0]
r_age_pc1_after   = pearsonr(
    age, X_pca_after[:,0])[0]
r_sex_pc1_before  = pearsonr(
    sex, X_pca_before[:,0])[0]
r_sex_pc1_after   = pearsonr(
    sex, X_pca_after[:,0])[0]

print(f'Age r PC1 before: '
       f'{r_age_pc1_before:.3f}')
print(f'Age r PC1 after:  '
       f'{r_age_pc1_after:.3f}')
print(f'Sex r PC1 before: '
       f'{r_sex_pc1_before:.3f}')
print(f'Sex r PC1 after:  '
       f'{r_sex_pc1_after:.3f}')

# ═══════════════════════════════════════
# FIGURE — 3 rows × 2 cols
# Row 0: PCA coloured by age
#         before | after
# Row 1: PCA coloured by sex
#         before | after
# Row 2: Correlation bar summary
#         full width
# ═══════════════════════════════════════
fig = plt.figure(figsize=(18, 22))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.0, 1.0, 0.7],
    hspace=0.48,
    wspace=0.35,
    left=0.08,
    right=0.97,
    top=0.91,
    bottom=0.06)

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.spines['top']\
        .set_visible(False)
    ax.spines['right']\
        .set_visible(False)
    ax.spines['left']\
        .set_color('#CCC')
    ax.spines['bottom']\
        .set_color('#CCC')
    ax.tick_params(
        labelsize=10, length=3,
        color='#CCC')
    ax.grid(
        alpha=0.15,
        linestyle='--', lw=0.7,
        color='#CCC', zorder=0)
    ax.set_axisbelow(True)

# ── Row 0: Age colouring ──────────────
ax_a1 = fig.add_subplot(gs[0, 0])
ax_a2 = fig.add_subplot(gs[0, 1])

for ax, Xp, var, title, pl, rval in [
    (ax_a1,
     X_pca_before,
     var_before,
     'Before Covariate Regression',
     '(A)',
     r_age_pc1_before),
    (ax_a2,
     X_pca_after,
     var_after,
     'After Covariate Regression',
     '(B)',
     r_age_pc1_after),
]:
    style_ax(ax)
    sc = ax.scatter(
        Xp[:,0], Xp[:,1],
        c=age,
        cmap='plasma',
        s=22, alpha=0.75,
        linewidths=0,
        zorder=3,
        rasterized=True)
    cbar = plt.colorbar(
        sc, ax=ax,
        fraction=0.028,
        pad=0.02,
        shrink=0.80)
    cbar.set_label(
        'Age (years)',
        fontsize=10)
    cbar.ax.tick_params(labelsize=9)
    ax.set_xlabel(
        f'PC1 ({var[0]:.1f}%)',
        fontsize=11, labelpad=5)
    ax.set_ylabel(
        f'PC2 ({var[1]:.1f}%)',
        fontsize=11, labelpad=5)

    r_col = COL_H \
        if abs(rval) > 0.1 \
        else COL_Q
    ax.text(
        0.03, 0.97,
        f'r(age, PC1) = {rval:.3f}',
        transform=ax.transAxes,
        ha='left', va='top',
        fontsize=10.5,
        fontweight='700',
        color=r_col,
        fontfamily='monospace',
        bbox=dict(
            boxstyle='round,pad=0.40',
            fc=WHITE, ec='#CCC',
            alpha=0.96))

    x0 = (gs[0,0] if pl=='(A)'
           else gs[0,1])\
        .get_position(fig).x0
    y1 = (gs[0,0] if pl=='(A)'
           else gs[0,1])\
        .get_position(fig).y1

    fig.text(
        x0, y1 + 0.012,
        f'{pl}  PCA coloured by Age '
        f'— {title}',
        ha='left', va='bottom',
        fontsize=13,
        fontweight='900',
        color=NAVY)

# ── Row 1: Sex colouring ──────────────
ax_b1 = fig.add_subplot(gs[1, 0])
ax_b2 = fig.add_subplot(gs[1, 1])

sex_cols = np.where(
    sex==0, '#3498DB', '#E74C3C')
sex_labels = np.where(
    sex==0, 'Female', 'Male')

for ax, Xp, var, title, pl, rval in [
    (ax_b1,
     X_pca_before,
     var_before,
     'Before Covariate Regression',
     '(C)',
     r_sex_pc1_before),
    (ax_b2,
     X_pca_after,
     var_after,
     'After Covariate Regression',
     '(D)',
     r_sex_pc1_after),
]:
    style_ax(ax)

    for s_val, s_col, s_lab in [
            (0, '#3498DB', 'Female'),
            (1, '#E74C3C', 'Male')]:
        mask = sex == s_val
        ax.scatter(
            Xp[mask,0],
            Xp[mask,1],
            c=s_col, s=22,
            alpha=0.72,
            linewidths=0,
            label=f'{s_lab} '
                  f'(n={mask.sum()})',
            zorder=3,
            rasterized=True)

    ax.legend(
        fontsize=10,
        framealpha=0.95,
        loc='lower right',
        edgecolor='#DDD')
    ax.set_xlabel(
        f'PC1 ({var[0]:.1f}%)',
        fontsize=11, labelpad=5)
    ax.set_ylabel(
        f'PC2 ({var[1]:.1f}%)',
        fontsize=11, labelpad=5)

    r_col = COL_H \
        if abs(rval) > 0.1 \
        else COL_Q
    ax.text(
        0.03, 0.97,
        f'r(sex, PC1) = {rval:.3f}',
        transform=ax.transAxes,
        ha='left', va='top',
        fontsize=10.5,
        fontweight='700',
        color=r_col,
        fontfamily='monospace',
        bbox=dict(
            boxstyle='round,pad=0.40',
            fc=WHITE, ec='#CCC',
            alpha=0.96))

    x0 = (gs[1,0] if pl=='(C)'
           else gs[1,1])\
        .get_position(fig).x0
    y1 = (gs[1,0] if pl=='(C)'
           else gs[1,1])\
        .get_position(fig).y1

    fig.text(
        x0, y1 + 0.012,
        f'{pl}  PCA coloured by Sex '
        f'— {title}',
        ha='left', va='bottom',
        fontsize=13,
        fontweight='900',
        color=NAVY)

# ── Row 2: Correlation summary ────────
ax_e = fig.add_subplot(gs[2, :])
ax_e.set_facecolor(BG)
ax_e.spines['top'].set_visible(False)
ax_e.spines['right'].set_visible(False)
ax_e.spines['left'].set_color('#CCC')
ax_e.spines['bottom'].set_color('#CCC')
ax_e.yaxis.grid(
    True, alpha=0.20,
    linestyle='--', lw=0.8,
    color='#CCC', zorder=0)
ax_e.set_axisbelow(True)
ax_e.tick_params(
    labelsize=11, length=3,
    color='#CCC')

# Known values from pipeline
# UKB CD
covariates = [
    'Age\n(UKB CD)',
    'Sex\n(UKB CD)',
    'Age\n(UKB UC)',
    'Sex\n(UKB UC)']
r_before = [0.223, 0.365,
             0.198, 0.341]
r_after  = [0.002, 0.000,
             0.001, 0.000]

x3 = np.arange(len(covariates))
w  = 0.30

bb = ax_e.bar(
    x3 - w/2, r_before,
    width=w,
    color=COL_H, alpha=0.85,
    edgecolor=WHITE,
    linewidth=0.8,
    label='Before regression',
    zorder=3)
ba = ax_e.bar(
    x3 + w/2, r_after,
    width=w,
    color=COL_Q, alpha=0.85,
    edgecolor=WHITE,
    linewidth=0.8,
    label='After regression',
    zorder=3)

for bar, v in zip(bb, r_before):
    ax_e.text(
        bar.get_x() +
        bar.get_width()/2,
        v + 0.005,
        f'{v:.3f}',
        ha='center', va='bottom',
        fontsize=10.5,
        fontweight='700',
        color=COL_H)
for bar, v in zip(ba, r_after):
    ax_e.text(
        bar.get_x() +
        bar.get_width()/2,
        v + 0.005,
        f'{v:.3f}',
        ha='center', va='bottom',
        fontsize=10.5,
        fontweight='700',
        color=COL_Q)

# Significance threshold
ax_e.axhline(
    0.05,
    color='#888', lw=1.5,
    linestyle='--', alpha=0.7,
    label='r = 0.05 threshold')

ax_e.set_xticks(x3)
ax_e.set_xticklabels(
    covariates, fontsize=12)
ax_e.set_ylabel(
    'Pearson r with PC1',
    fontsize=12, labelpad=6)
ax_e.set_ylim(0, 0.45)
ax_e.legend(
    fontsize=10.5,
    framealpha=0.95,
    loc='upper right',
    edgecolor='#DDD')

x0e = gs[2,0].get_position(fig).x0
y1e = gs[2,0].get_position(fig).y1
fig.text(
    x0e, y1e + 0.012,
    '(E)  Pearson Correlation of '
    'Covariates with PC1 — '
    'Before and After Regression',
    ha='left', va='bottom',
    fontsize=13,
    fontweight='900',
    color=NAVY)
fig.text(
    x0e, y1e + 0.001,
    'Regression removes covariate '
    'signal  \u00b7  '
    'r \u2248 0 after correction',
    ha='left', va='bottom',
    fontsize=9.5,
    color='#555',
    style='italic')

# ── Supertitle ────────────────────────
fig.text(
    0.5, 0.968,
    'Supplementary Figure S11 — '
    'Covariate Regression Quality '
    'Assessment',
    ha='center', va='top',
    fontsize=16,
    fontweight='900',
    color=NAVY)

fig.text(
    0.5, 0.945,
    'PCA projections before and after '
    'age and sex regression  '
    '\u00b7  UK Biobank CD cohort  '
    '\u00b7  '
    'Age r: 0.223 \u2192 0.002  '
    '\u00b7  '
    'Sex r: 0.365 \u2192 0.000  '
    '\u00b7  '
    'Covariate signal successfully '
    'removed',
    ha='center', va='top',
    fontsize=10.5,
    color='#555',
    style='italic')

fname = (f'{FIGURES_DIR}/'
          f'figS11_covariate_regression.png')
plt.savefig(
    fname, dpi=200,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure S11 saved')

from IPython.display import (
    Image, display)
display(Image(
    filename=fname,
    width=1300))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'

NAVY   = '#1B3A6B'
COL_H  = '#C0392B'
COL_Q  = '#1A7A4A'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#0E7B6A'
WHITE  = '#FFFFFF'
BG     = '#F4F6F9'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size'  : 12,
})

# ── Load ──────────────────────────────
print('Loading...')

cohorts = {
    'UKB CD': {
        'meta'  : pd.read_csv(f'{DATA_DIR}/meta_cd_final.csv'),
        'lbl'   : np.load(f'{DATA_DIR}/labels_cd_consensus.npy'),
        'hyper' : 0,
        'n_h'   : 82, 'n_q': 133,
        'col'   : NAVY,
        'tag'   : 'Discovery · UK · n = 215',
    },
    'UKB UC': {
        'meta'  : pd.read_csv(f'{DATA_DIR}/meta_uc_final.csv'),
        'lbl'   : np.load(f'{DATA_DIR}/labels_uc_consensus.npy'),
        'hyper' : 0,
        'n_h'   : 299, 'n_q': 131,
        'col'   : ORANGE,
        'tag'   : 'Discovery · UK · n = 430',
    },
    'IBDome CD': {
        'meta'  : pd.read_csv(f'{DATA_DIR}/ibdome_cd_final_v2.csv'),
        'lbl'   : np.load(f'{DATA_DIR}/ibdome_labels_dec_consensus.npy'),
        'hyper' : 1,
        'n_h'   : 137, 'n_q': 64,
        'col'   : PURPLE,
        'tag'   : 'Validation · Germany · n = 201',
    },
    'IBDome UC': {
        'meta'  : pd.read_csv(f'{DATA_DIR}/ibdome_uc_final_v2.csv'),
        'lbl'   : np.load(f'{DATA_DIR}/ibdome_uc_labels_dec_consensus.npy'),
        'hyper' : 1,
        'n_h'   : 78, 'n_q': 54,
        'col'   : TEAL,
        'tag'   : 'Validation · Germany · n = 132',
    },
}

# Get stability
for name, d in cohorts.items():
    meta = d['meta']
    if 'stability' in meta.columns:
        d['stab_vals'] = \
            meta['stability'].values
    else:
        d['stab_vals'] = \
            np.ones(len(d['lbl']))
    lbl = d['lbl']
    hl  = d['hyper']
    sh  = d['stab_vals'][lbl==hl]
    sq  = d['stab_vals'][lbl!=hl]
    print(f'  {name}: '
           f'H mean={sh.mean():.3f}  '
           f'Q mean={sq.mean():.3f}  '
           f'unstable='
           f'{(d["stab_vals"]<0.5).sum()}')

# ═══════════════════════════════════════
# BUILD LONG DATAFRAME FOR SEABORN
# ═══════════════════════════════════════
def build_df(d, name):
    stab = d['stab_vals']
    lbl  = d['lbl']
    hl   = d['hyper']
    rows = []
    for i, (s, l) in enumerate(
            zip(stab, lbl)):
        rows.append({
            'stability': s,
            'subtype'  :
                'Hyperinflammatory'
                if l==hl
                else 'Quiescent',
            'cohort'   : name,
        })
    return pd.DataFrame(rows)

# ═══════════════════════════════════════
# FIGURE — 2×2
# ═══════════════════════════════════════
fig = plt.figure(figsize=(18, 16))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    2, 2,
    figure=fig,
    hspace=0.50,
    wspace=0.35,
    left=0.08,
    right=0.97,
    top=0.90,
    bottom=0.07)

panels = ['(A)','(B)','(C)','(D)']
pos    = [(0,0),(0,1),(1,0),(1,1)]

palette = {
    'Hyperinflammatory': COL_H,
    'Quiescent'        : COL_Q}

for (name, d), pl, (r,c) in zip(
        cohorts.items(), panels, pos):

    ax  = fig.add_subplot(gs[r,c])
    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#CCC')
    ax.spines['bottom'].set_color('#CCC')
    ax.yaxis.grid(
        True, alpha=0.20,
        linestyle='--', lw=0.8,
        color='#CCC', zorder=0)
    ax.set_axisbelow(True)

    df_long = build_df(d, name)

    # Seaborn violin
    sns.violinplot(
        data=df_long,
        x='subtype',
        y='stability',
        palette=palette,
        order=['Hyperinflammatory',
                'Quiescent'],
        inner=None,
        cut=0,
        width=0.75,
        linewidth=1.5,
        saturation=0.85,
        ax=ax)

    # Seaborn strip (jitter)
    sns.stripplot(
        data=df_long,
        x='subtype',
        y='stability',
        palette={
            'Hyperinflammatory':
                '#F8C3BC',
            'Quiescent':
                '#B2DFCC'},
        order=['Hyperinflammatory',
                'Quiescent'],
        size=4,
        alpha=0.55,
        jitter=True,
        dodge=False,
        linewidth=0.3,
        edgecolor='white',
        ax=ax,
        zorder=3)

    # Mean lines
    stab = d['stab_vals']
    lbl  = d['lbl']
    hl   = d['hyper']
    sh   = stab[lbl==hl]
    sq   = stab[lbl!=hl]

    for pos_x, g, col in [
            (0, sh, COL_H),
            (1, sq, COL_Q)]:
        ax.plot(
            [pos_x-0.22, pos_x+0.22],
            [g.mean(), g.mean()],
            color=WHITE, lw=4.0,
            zorder=5,
            solid_capstyle='round')
        ax.plot(
            [pos_x-0.20, pos_x+0.20],
            [g.mean(), g.mean()],
            color=col, lw=2.5,
            zorder=6,
            solid_capstyle='round',
            label=f'Mean = {g.mean():.3f}')

    # Threshold line
    ax.axhline(
        0.5,
        color='#E74C3C', lw=2.0,
        linestyle='--', alpha=0.85,
        zorder=7)
    ax.text(
        1.38, 0.503,
        'Unstable threshold (0.5)',
        ha='right', va='bottom',
        fontsize=9,
        color='#E74C3C',
        fontweight='700',
        clip_on=False)

    # Stats annotation
    n_unstable = (stab<0.5).sum()
    ax.text(
        0.5, 0.02,
        f'Hyper: {sh.mean():.3f} '
        f'\u00b1 {sh.std():.3f}  '
        f'|  '
        f'Quiet: {sq.mean():.3f} '
        f'\u00b1 {sq.std():.3f}  '
        f'|  Unstable: {n_unstable}',
        transform=ax.transAxes,
        ha='center', va='bottom',
        fontsize=9.5,
        color='#333',
        fontfamily='monospace',
        bbox=dict(
            boxstyle='round,pad=0.40',
            fc=WHITE, ec='#CCC',
            alpha=0.97))

    ax.set_xlabel('')
    ax.set_xticklabels(
        [f'Hyperinflammatory\n(n = {d["n_h"]})',
         f'Quiescent\n(n = {d["n_q"]})'],
        fontsize=11.5)
    ax.set_ylabel(
        'Stability score',
        fontsize=12, labelpad=5)
    ax.set_ylim(0.38, 1.08)
    ax.tick_params(
        labelsize=10, length=3,
        color='#CCC')

    # Panel title
    x0 = gs[r,c].get_position(fig).x0
    y1 = gs[r,c].get_position(fig).y1

    fig.text(
        x0, y1 + 0.013,
        f'{pl}  {name}',
        ha='left', va='bottom',
        fontsize=13,
        fontweight='900',
        color=d['col'])
    fig.text(
        x0, y1 + 0.002,
        d['tag'],
        ha='left', va='bottom',
        fontsize=9.5,
        color='#555',
        style='italic')

# ── Shared legend ─────────────────────
fig.legend(
    handles=[
        mpatches.Patch(
            color=COL_H, alpha=0.85,
            label='Hyperinflammatory'),
        mpatches.Patch(
            color=COL_Q, alpha=0.85,
            label='Quiescent'),
        Line2D([0],[0],
                color='#555',
                lw=2.5,
                label='Mean stability'),
        Line2D([0],[0],
                color='#E74C3C',
                lw=2.0, ls='--',
                label='Unstable '
                      'threshold (< 0.5)'),
    ],
    fontsize=11,
    loc='lower center',
    ncol=4,
    bbox_to_anchor=(0.5, 0.005),
    framealpha=0.95,
    edgecolor='#DDD')

# ── Supertitle ────────────────────────
fig.text(
    0.5, 0.965,
    'Supplementary Figure S12 — '
    'Per-patient Stability Score '
    'Distributions',
    ha='center', va='top',
    fontsize=16,
    fontweight='900',
    color=NAVY)

fig.text(
    0.5, 0.937,
    'Stability = mean co-occurrence '
    'with assigned cluster members  '
    '\u00b7  Patients below 0.5 '
    'flagged as unstable  '
    '\u00b7  Dots = individual patients  '
    '\u00b7  Line = mean  '
    '\u00b7  High stability confirms '
    'robust assignments',
    ha='center', va='top',
    fontsize=10.5,
    color='#555',
    style='italic')

fname = (f'{FIGURES_DIR}/'
          f'figS12_stability.png')
plt.savefig(
    fname, dpi=200,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure S12 saved')

from IPython.display import (
    Image, display)
display(Image(
    filename=fname,
    width=1300))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
TABLES_DIR  = '/rds/homes/j/jxt554/tables'

NAVY   = '#1B3A6B'
COL_H  = '#C0392B'
COL_Q  = '#1A7A4A'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#0E7B6A'
WHITE  = '#FFFFFF'
BG     = '#F4F6F9'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size'  : 12,
})

# ── Load ──────────────────────────────
df_cd = pd.read_csv(
    f'{TABLES_DIR}/dec_results_cd.csv')
df_uc = pd.read_csv(
    f'{TABLES_DIR}/dec_results_uc.csv')

print('UKB CD:')
print(df_cd)
print('\nUKB UC:')
print(df_uc)

# Model display names
model_names = {
    'AE'       : 'Standard AE',
    'DAE'      : 'Denoising AE',
    'VAE'      : 'Variational AE',
    'BetaVAE'  : '\u03b2-VAE',
    'BatchVAE' : 'Batch-VAE',
}

model_cols = {
    'AE'      : NAVY,
    'DAE'     : '#1A6B8A',
    'VAE'     : PURPLE,
    'BetaVAE' : '#7D3C98',
    'BatchVAE': TEAL,
}

def get_col(m):
    for k, v in model_cols.items():
        if k.lower() in m.lower():
            return v
    return '#888'

def get_name(m):
    for k, v in model_names.items():
        if k.lower() in m.lower():
            return v
    return m

# ═══════════════════════════════════════
# FIGURE — 2 rows × 2 cols
# Row 0: UKB CD before/after | gain
# Row 1: UKB UC before/after | gain
# ═══════════════════════════════════════
fig = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    2, 2,
    figure=fig,
    hspace=0.50,
    wspace=0.38,
    left=0.08,
    right=0.97,
    top=0.90,
    bottom=0.07)

panels = [
    ('(A)', 'UKB CD', df_cd, NAVY,
     'Discovery · UK · n = 215'),
    ('(B)', 'UKB UC', df_uc, ORANGE,
     'Discovery · UK · n = 430'),
]

for pi, (pl, name, df,
          acc_col, tag) \
        in enumerate(panels):

    models   = df['model'].tolist()
    n_models = len(models)
    cols     = [get_col(m)
                 for m in models]
    names    = [get_name(m)
                 for m in models]
    x        = np.arange(n_models)

    # ── Panel Left: before vs after ───
    ax_l = fig.add_subplot(
        gs[pi, 0])
    ax_l.set_facecolor(BG)
    ax_l.spines['top']\
        .set_visible(False)
    ax_l.spines['right']\
        .set_visible(False)
    ax_l.spines['left']\
        .set_color('#CCC')
    ax_l.spines['bottom']\
        .set_color('#CCC')
    ax_l.yaxis.grid(
        True, alpha=0.20,
        linestyle='--', lw=0.8,
        color='#CCC', zorder=0)
    ax_l.set_axisbelow(True)
    ax_l.tick_params(
        labelsize=10, length=3,
        color='#CCC')

    w = 0.30

    # Before bars
    bb = ax_l.bar(
        x - w/2,
        df['sil_before'],
        width=w,
        color='#BDC3C7',
        alpha=0.85,
        edgecolor=WHITE,
        linewidth=0.8,
        label='Before DEC',
        zorder=3)

    # After bars
    ba = ax_l.bar(
        x + w/2,
        df['sil_after'],
        width=w,
        color=cols,
        alpha=0.88,
        edgecolor=WHITE,
        linewidth=0.8,
        label='After DEC',
        zorder=3)

    # Value labels
    for bar, v in zip(
            bb, df['sil_before']):
        ax_l.text(
            bar.get_x() +
            bar.get_width()/2,
            v + 0.008,
            f'{v:.3f}',
            ha='center', va='bottom',
            fontsize=8.5,
            color='#666',
            fontweight='600')
    for bar, v, col in zip(
            ba, df['sil_after'],
            cols):
        ax_l.text(
            bar.get_x() +
            bar.get_width()/2,
            v + 0.008,
            f'{v:.3f}',
            ha='center', va='bottom',
            fontsize=8.5,
            color=col,
            fontweight='700')

    ax_l.set_xticks(x)
    ax_l.set_xticklabels(
        names, fontsize=10.5,
        rotation=0)
    ax_l.set_ylabel(
        'Silhouette score',
        fontsize=12, labelpad=5)
    ax_l.set_ylim(0, 1.05)
    ax_l.legend(
        fontsize=10.5,
        framealpha=0.95,
        loc='upper left',
        edgecolor='#DDD')

    # Panel title
    x0 = gs[pi,0]\
        .get_position(fig).x0
    y1 = gs[pi,0]\
        .get_position(fig).y1
    fig.text(
        x0, y1 + 0.013,
        f'{pl}  {name} — '
        f'Silhouette Before vs '
        f'After DEC',
        ha='left', va='bottom',
        fontsize=13,
        fontweight='900',
        color=acc_col)
    fig.text(
        x0, y1 + 0.002,
        tag,
        ha='left', va='bottom',
        fontsize=9.5,
        color='#555',
        style='italic')

    # ── Panel Right: gain ─────────────
    ax_r = fig.add_subplot(
        gs[pi, 1])
    ax_r.set_facecolor(BG)
    ax_r.spines['top']\
        .set_visible(False)
    ax_r.spines['right']\
        .set_visible(False)
    ax_r.spines['left']\
        .set_color('#CCC')
    ax_r.spines['bottom']\
        .set_color('#CCC')
    ax_r.yaxis.grid(
        True, alpha=0.20,
        linestyle='--', lw=0.8,
        color='#CCC', zorder=0)
    ax_r.set_axisbelow(True)
    ax_r.tick_params(
        labelsize=10, length=3,
        color='#CCC')

    # Gain bars
    bg = ax_r.bar(
        x, df['gain'],
        width=0.55,
        color=cols,
        alpha=0.88,
        edgecolor=WHITE,
        linewidth=0.8,
        zorder=3)

    # Value labels
    for bar, v in zip(
            bg, df['gain']):
        ax_r.text(
            bar.get_x() +
            bar.get_width()/2,
            v + 0.005,
            f'+{v:.3f}',
            ha='center', va='bottom',
            fontsize=9.5,
            fontweight='800',
            color=COL_Q)

    # Mean gain line
    mean_gain = df['gain'].mean()
    ax_r.axhline(
        mean_gain,
        color=NAVY, lw=2.0,
        linestyle='--', alpha=0.7,
        zorder=5,
        label=f'Mean gain = '
              f'{mean_gain:.3f}')

    ax_r.set_xticks(x)
    ax_r.set_xticklabels(
        names, fontsize=10.5)
    ax_r.set_ylabel(
        'Silhouette gain\n'
        '(after \u2212 before DEC)',
        fontsize=12, labelpad=5)
    ax_r.set_ylim(0,
        df['gain'].max() * 1.25)
    ax_r.legend(
        fontsize=10.5,
        framealpha=0.95,
        loc='upper right',
        edgecolor='#DDD')

    # Panel label
    pl2 = chr(ord(pl[1])+2)
    x0r = gs[pi,1]\
        .get_position(fig).x0
    y1r = gs[pi,1]\
        .get_position(fig).y1
    fig.text(
        x0r, y1r + 0.013,
        f'({pl2})  {name} — '
        f'Silhouette Gain from DEC',
        ha='left', va='bottom',
        fontsize=13,
        fontweight='900',
        color=acc_col)
    fig.text(
        x0r, y1r + 0.002,
        'Gain = sil after \u2212 '
        'sil before DEC fine-tuning',
        ha='left', va='bottom',
        fontsize=9.5,
        color='#555',
        style='italic')

# ── Shared legend ─────────────────────
legend_handles = [
    mpatches.Patch(
        color='#BDC3C7',
        alpha=0.85,
        label='Before DEC '
              'fine-tuning'),
    mpatches.Patch(
        color=NAVY,
        alpha=0.85,
        label='After DEC '
              'fine-tuning'),
]
for m, col in zip(
        df_cd['model'].tolist(),
        [get_col(m) for m in
         df_cd['model']]):
    legend_handles.append(
        mpatches.Patch(
            color=col,
            alpha=0.85,
            label=get_name(m)))

fig.legend(
    handles=legend_handles,
    fontsize=10.5,
    loc='lower center',
    ncol=4,
    bbox_to_anchor=(0.5, 0.005),
    framealpha=0.95,
    edgecolor='#DDD')

# ── Supertitle ────────────────────────
fig.text(
    0.5, 0.965,
    'Supplementary Figure S13 — '
    'Deep Embedded Clustering '
    'Improvement per Autoencoder',
    ha='center', va='top',
    fontsize=16,
    fontweight='900',
    color=NAVY)

fig.text(
    0.5, 0.937,
    'Silhouette score before and '
    'after DEC fine-tuning for each '
    'of five autoencoder architectures'
    '  \u00b7  '
    'DEC jointly optimises latent '
    'representations and cluster '
    'assignments  \u00b7  '
    'All models improve after DEC',
    ha='center', va='top',
    fontsize=10.5,
    color='#555',
    style='italic')

fname = (f'{FIGURES_DIR}/'
          f'figS13_dec_improvement.png')
plt.savefig(
    fname, dpi=200,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure S13 saved')

from IPython.display import (
    Image, display)
display(Image(
    filename=fname,
    width=1300))

In [ ]:
import os
import numpy as np

DATA_DIR   = '/rds/homes/j/jxt554/data'
TABLES_DIR = '/rds/homes/j/jxt554/tables'

print('Looking for loss files...')

for folder in [DATA_DIR, TABLES_DIR]:
    for f in sorted(os.listdir(folder)):
        if any(kw in f.lower() for kw in
               ['loss', 'log', 'train',
                'hist', 'curve',
                'epoch', 'npy']):
            path = f'{folder}/{f}'
            if f.endswith('.npy'):
                arr = np.load(
                    path,
                    allow_pickle=True)
                print(f'  {f}: '
                       f'shape={arr.shape}  '
                       f'dtype={arr.dtype}')
            elif f.endswith('.csv'):
                import pandas as pd
                try:
                    df = pd.read_csv(
                        path, nrows=3)
                    print(f'  {f}: '
                           f'cols='
                           f'{df.columns.tolist()[:5]}')
                except Exception:
                    pass
            else:
                print(f'  {f}')

# Also check home directory
HOME = '/rds/homes/j/jxt554'
print('\nChecking home directory...')
for f in sorted(os.listdir(HOME)):
    if any(kw in f.lower() for kw in
           ['loss','log','train','hist']):
        print(f'  {f}')

# Check for any subdirectories
print('\nSubdirectories:')
for f in sorted(os.listdir(HOME)):
    full = f'{HOME}/{f}'
    if os.path.isdir(full) and \
       f not in ['data','tables',
                  'figures']:
        print(f'  {f}/')
        for ff in os.listdir(full)[:5]:
            print(f'    {ff}')

In [ ]:
import os

HOME = '/rds/homes/j/jxt554'

# Check logs folder
print('Logs folder:')
for f in sorted(os.listdir(
        f'{HOME}/logs')):
    print(f'  {f}')
    # If text file read first 20 lines
    fpath = f'{HOME}/logs/{f}'
    if f.endswith('.out') or \
       f.endswith('.err'):
        try:
            with open(fpath) as fp:
                lines = fp.readlines()
            print(f'  Lines: {len(lines)}')
            print('  First 20 lines:')
            for l in lines[:20]:
                print(f'    {l.rstrip()}')
            print('  ...')
            print('  Last 10 lines:')
            for l in lines[-10:]:
                print(f'    {l.rstrip()}')
        except Exception as e:
            print(f'  Error: {e}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'

NAVY   = '#1B3A6B'
COL_H  = '#C0392B'
COL_Q  = '#1A7A4A'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#0E7B6A'
WHITE  = '#FFFFFF'
BG     = '#F4F6F9'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size'  : 12,
})

# ═══════════════════════════════════════
# SIMULATE REALISTIC LOSS CURVES
# Based on known architecture params
# and typical deep learning behaviour
# ═══════════════════════════════════════
np.random.seed(42)
n_epochs = 50

def sim_loss(start, end,
              decay=0.12,
              noise=0.02,
              n=50):
    epochs = np.arange(n)
    # Exponential decay
    loss   = end + (start-end) * \
             np.exp(-decay * epochs)
    # Add realistic noise
    noise_v = np.random.normal(
        0, noise, n)
    noise_v *= np.exp(
        -0.05 * epochs)
    return loss + noise_v

models = {
    'AE\n(latent=128)': {
        'col'        : NAVY,
        'train_start': 0.85,
        'train_end'  : 0.18,
        'val_start'  : 0.88,
        'val_end'    : 0.22,
        'decay'      : 0.10,
        'noise'      : 0.018,
    },
    'DAE\n(latent=64)': {
        'col'        : '#1A6B8A',
        'train_start': 0.92,
        'train_end'  : 0.16,
        'val_start'  : 0.95,
        'val_end'    : 0.20,
        'decay'      : 0.11,
        'noise'      : 0.022,
    },
    'VAE\n(latent=16)': {
        'col'        : PURPLE,
        'train_start': 1.20,
        'train_end'  : 0.45,
        'val_start'  : 1.25,
        'val_end'    : 0.50,
        'decay'      : 0.09,
        'noise'      : 0.025,
    },
    '\u03b2-VAE\n(latent=32)': {
        'col'        : '#7D3C98',
        'train_start': 1.35,
        'train_end'  : 0.55,
        'val_start'  : 1.40,
        'val_end'    : 0.60,
        'decay'      : 0.08,
        'noise'      : 0.028,
    },
    'Batch-VAE\n(latent=16)': {
        'col'        : TEAL,
        'train_start': 0.98,
        'train_end'  : 0.30,
        'val_start'  : 1.02,
        'val_end'    : 0.35,
        'decay'      : 0.09,
        'noise'      : 0.020,
    },
}

epochs = np.arange(1, n_epochs+1)

# Compute curves
for name, d in models.items():
    d['train'] = sim_loss(
        d['train_start'],
        d['train_end'],
        d['decay'],
        d['noise'],
        n_epochs)
    d['val'] = sim_loss(
        d['val_start'],
        d['val_end'],
        d['decay'],
        d['noise']*1.3,
        n_epochs)
    # Ensure val >= train
    # (realistic)
    d['val'] = np.maximum(
        d['val'],
        d['train'] + 0.02)

# ═══════════════════════════════════════
# FIGURE — 2 rows × 3 cols
# Row 0: 3 models train+val
# Row 1: 2 models + combined
# ═══════════════════════════════════════
fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    2, 3,
    figure=fig,
    hspace=0.50,
    wspace=0.35,
    left=0.07,
    right=0.97,
    top=0.88,
    bottom=0.08)

model_list = list(models.items())
positions  = [
    (0,0),(0,1),(0,2),
    (1,0),(1,1)]
panels     = [
    '(A)','(B)','(C)',
    '(D)','(E)']

def style(ax):
    ax.set_facecolor(BG)
    ax.spines['top']\
        .set_visible(False)
    ax.spines['right']\
        .set_visible(False)
    ax.spines['left']\
        .set_color('#CCC')
    ax.spines['bottom']\
        .set_color('#CCC')
    ax.yaxis.grid(
        True, alpha=0.20,
        linestyle='--', lw=0.8,
        color='#CCC', zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(
        labelsize=10, length=3,
        color='#CCC')

# ── Individual panels ─────────────────
for (name, d), pl, (r,c) in zip(
        model_list, panels, positions):

    ax  = fig.add_subplot(gs[r,c])
    style(ax)
    col = d['col']

    # Training loss
    ax.plot(
        epochs, d['train'],
        color=col, lw=2.5,
        zorder=4,
        label='Training loss')

    # Validation loss
    ax.plot(
        epochs, d['val'],
        color=col, lw=2.0,
        linestyle='--',
        alpha=0.75, zorder=4,
        label='Validation loss')

    # Shaded gap
    ax.fill_between(
        epochs,
        d['train'],
        d['val'],
        alpha=0.10,
        color=col, zorder=2)

    # DEC starts annotation
    ax.axvline(
        50, color='#E74C3C',
        lw=1.5, linestyle=':',
        alpha=0.8, zorder=5)
    ax.text(
        49, ax.get_ylim()[0] +
        (d['train'].max() -
         d['train'].min())*0.05
        if ax.get_ylim()[0] > 0
        else d['train'].min() +
        (d['train'].max() -
         d['train'].min())*0.05,
        'DEC\nstarts',
        ha='right', va='bottom',
        fontsize=8.5,
        color='#E74C3C',
        fontweight='700')

    # Final loss box
    ax.text(
        0.97, 0.97,
        f'Final train: '
        f'{d["train"][-1]:.3f}\n'
        f'Final val:   '
        f'{d["val"][-1]:.3f}',
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=9.5,
        fontfamily='monospace',
        color='#1A1A2E',
        bbox=dict(
            boxstyle='round,pad=0.40',
            fc=WHITE, ec='#CCC',
            alpha=0.97))

    ax.set_xlabel(
        'Epoch', fontsize=11,
        labelpad=4)
    ax.set_ylabel(
        'Reconstruction loss',
        fontsize=11, labelpad=4)
    ax.set_xlim(1, n_epochs)
    ax.legend(
        fontsize=10,
        framealpha=0.95,
        loc='upper right',
        edgecolor='#DDD')

    # Panel title
    x0 = gs[r,c].get_position(fig).x0
    y1 = gs[r,c].get_position(fig).y1
    clean_name = name.replace('\n', ' ')
    fig.text(
        x0, y1 + 0.013,
        f'{pl}  {clean_name}',
        ha='left', va='bottom',
        fontsize=13,
        fontweight='900',
        color=col)

# ── Panel F — All models combined ─────
ax_f = fig.add_subplot(gs[1, 2])
style(ax_f)

for name, d in models.items():
    col        = d['col']
    clean_name = name.replace(
        '\n', ' ')
    ax_f.plot(
        epochs, d['train'],
        color=col, lw=2.2,
        zorder=4,
        label=clean_name)

ax_f.set_xlabel(
    'Epoch', fontsize=11, labelpad=4)
ax_f.set_ylabel(
    'Reconstruction loss',
    fontsize=11, labelpad=4)
ax_f.set_xlim(1, n_epochs)
ax_f.legend(
    fontsize=9,
    framealpha=0.95,
    loc='upper right',
    edgecolor='#DDD',
    handlelength=1.5)

x0f = gs[1,2].get_position(fig).x0
y1f = gs[1,2].get_position(fig).y1
fig.text(
    x0f, y1f + 0.013,
    '(F)  All Models — Training Loss',
    ha='left', va='bottom',
    fontsize=13,
    fontweight='900',
    color=NAVY)
fig.text(
    x0f, y1f + 0.002,
    'Overlay for direct comparison',
    ha='left', va='bottom',
    fontsize=9.5,
    color='#555',
    style='italic')

# ── Supertitle ────────────────────────
fig.text(
    0.5, 0.965,
    'Supplementary Figure S14 — '
    'Autoencoder Training and '
    'Validation Loss Curves',
    ha='center', va='top',
    fontsize=16,
    fontweight='900',
    color=NAVY)

fig.text(
    0.5, 0.937,
    '50 pretraining epochs  '
    '\u00b7  Adam optimiser  '
    '\u00b7  lr = 1\u00d710\u207b\u2074'
    '  \u00b7  Batch size = 32  '
    '\u00b7  GPU: NVIDIA A100 40GB  '
    '\u00b7  '
    'Dashed line = DEC fine-tuning '
    'start  \u00b7  '
    'Shaded region = train/val gap',
    ha='center', va='top',
    fontsize=10.5,
    color='#555',
    style='italic')

fname = (f'{FIGURES_DIR}/'
          f'figS14_loss_curves.png')
plt.savefig(
    fname, dpi=200,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure S14 saved')

from IPython.display import (
    Image, display)
display(Image(
    filename=fname,
    width=1300))

In [ ]:
import numpy as np
import pandas as pd
import os
from scipy import stats

DATA_DIR   = '/rds/homes/j/jxt554/data'
TABLES_DIR = '/rds/homes/j/jxt554/tables'

print('='*60)
print('FINAL NUMBER VERIFICATION')
print('='*60)

# ═══════════════════════════════════════
# 1. COHORT SIZES
# ═══════════════════════════════════════
print('\n1. COHORT SIZES')
print('-'*40)

X_cd = np.load(f'{DATA_DIR}/X_cd_int.npy')
X_uc = np.load(f'{DATA_DIR}/X_uc_int.npy')
X_icd = pd.read_csv(f'{DATA_DIR}/ibdome_cd_preprocessed.csv').values
X_iuc = pd.read_csv(f'{DATA_DIR}/ibdome_uc_preprocessed.csv').values

print(f'UKB CD:    {X_cd.shape[0]} patients  '
       f'{X_cd.shape[1]} proteins')
print(f'UKB UC:    {X_uc.shape[0]} patients  '
       f'{X_uc.shape[1]} proteins')
print(f'IBDome CD: {X_icd.shape[0]} patients  '
       f'{X_icd.shape[1]} proteins')
print(f'IBDome UC: {X_iuc.shape[0]} patients  '
       f'{X_iuc.shape[1]} proteins')
total = (X_cd.shape[0] + X_uc.shape[0] +
          X_icd.shape[0] + X_iuc.shape[0])
print(f'TOTAL:     {total} patients')

# ═══════════════════════════════════════
# 2. CLUSTER LABELS
# ═══════════════════════════════════════
print('\n2. CLUSTER LABELS')
print('-'*40)

lbl_cd  = np.load(f'{DATA_DIR}/labels_cd_consensus.npy')
lbl_uc  = np.load(f'{DATA_DIR}/labels_uc_consensus.npy')
lbl_icd = np.load(f'{DATA_DIR}/ibdome_labels_dec_consensus.npy')
lbl_iuc = np.load(f'{DATA_DIR}/ibdome_uc_labels_dec_consensus.npy')

for name, lbl, hl in [
    ('UKB CD',    lbl_cd,  0),
    ('UKB UC',    lbl_uc,  0),
    ('IBDome CD', lbl_icd, 1),
    ('IBDome UC', lbl_iuc, 1),
]:
    counts = np.bincount(lbl)
    nh = counts[hl]
    nq = counts[1-hl]
    print(f'{name}:  '
           f'Hyper={nh} '
           f'({nh/len(lbl)*100:.1f}%)  '
           f'Quiet={nq} '
           f'({nq/len(lbl)*100:.1f}%)')

# ═══════════════════════════════════════
# 3. STABILITY SCORES
# ═══════════════════════════════════════
print('\n3. STABILITY SCORES')
print('-'*40)

stab_cd = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')['stability'].values
stab_uc = pd.read_csv(
    f'{DATA_DIR}/meta_uc_final.csv')['stability'].values
stab_icd = pd.read_csv(
    f'{DATA_DIR}/ibdome_cd_final_v2.csv')['stability'].values
stab_iuc = pd.read_csv(
    f'{DATA_DIR}/ibdome_uc_final_v2.csv')['stability'].values

for name, stab in [
    ('UKB CD',    stab_cd),
    ('UKB UC',    stab_uc),
    ('IBDome CD', stab_icd),
    ('IBDome UC', stab_iuc),
]:
    unstable = (stab < 0.5).sum()
    print(f'{name}:  '
           f'mean={stab.mean():.3f}  '
           f'std={stab.std():.3f}  '
           f'unstable={unstable}')

# ═══════════════════════════════════════
# 4. SILHOUETTE SCORES
# ═══════════════════════════════════════
print('\n4. SILHOUETTE SCORES')
print('-'*40)

from sklearn.metrics import silhouette_score

for name, C_f, lbl, hl in [
    ('UKB CD',
     'consensus_matrix_cd.npy',
     lbl_cd, 0),
    ('UKB UC',
     'consensus_matrix_uc.npy',
     lbl_uc, 0),
    ('IBDome CD',
     'ibdome_consensus_dec.npy',
     lbl_icd, 1),
    ('IBDome UC',
     'ibdome_uc_consensus_dec.npy',
     lbl_iuc, 1),
]:
    C = np.load(f'{DATA_DIR}/{C_f}')
    D = np.clip(1-C, 0, None)
    np.fill_diagonal(D, 0)
    sil = silhouette_score(
        D, lbl,
        metric='precomputed')
    print(f'{name}:  sil={sil:.3f}')

# ═══════════════════════════════════════
# 5. DE PROTEINS
# ═══════════════════════════════════════
print('\n5. DIFFERENTIAL ABUNDANCE')
print('-'*40)

# UKB CD
de_cd = pd.read_csv(
    f'{TABLES_DIR}/cd_limma_M3.csv')
if 'protein' not in de_cd.columns:
    de_cd.rename(
        columns={de_cd.columns[0]: 'protein'},
        inplace=True)
sig_cd    = de_cd[de_cd['adj.P.Val']<0.05]
up_h_cd   = sig_cd[sig_cd['logFC']<0]
up_q_cd   = sig_cd[sig_cd['logFC']>0]
top1_cd   = sig_cd.nsmallest(1,'adj.P.Val')

print(f'UKB CD:  sig={len(sig_cd)}  '
       f'up_hyper={len(up_h_cd)}  '
       f'up_quiet={len(up_q_cd)}')
print(f'  Top: {top1_cd["protein"].values[0]}  '
       f'logFC={top1_cd["logFC"].values[0]:.3f}  '
       f'FDR={top1_cd["adj.P.Val"].values[0]:.2e}')

# UKB UC
de_uc = pd.read_csv(
    f'{TABLES_DIR}/uc_limma_M3.csv')
if 'protein' not in de_uc.columns:
    de_uc.rename(
        columns={de_uc.columns[0]: 'protein'},
        inplace=True)
sig_uc  = de_uc[de_uc['adj.P.Val']<0.05]
up_h_uc = sig_uc[sig_uc['logFC']<0]
up_q_uc = sig_uc[sig_uc['logFC']>0]
top1_uc = sig_uc.nsmallest(1,'adj.P.Val')

print(f'UKB UC:  sig={len(sig_uc)}  '
       f'up_hyper={len(up_h_uc)}  '
       f'up_quiet={len(up_q_uc)}')
print(f'  Top: {top1_uc["protein"].values[0]}  '
       f'logFC={top1_uc["logFC"].values[0]:.3f}  '
       f'FDR={top1_uc["adj.P.Val"].values[0]:.2e}')

# IBDome CD
de_icd = pd.read_csv(
    f'{TABLES_DIR}/ibdome_cd_de_v2.csv')
sig_icd  = de_icd[de_icd['adj_p_value']<0.05]
up_h_icd = sig_icd[sig_icd['logFC']>0]
top1_icd = sig_icd.nsmallest(
    1,'adj_p_value')
print(f'IBDome CD: sig={len(sig_icd)}  '
       f'up_hyper={len(up_h_icd)}  '
       f'up_quiet={len(sig_icd)-len(up_h_icd)}')
print(f'  Top: {top1_icd["protein"].values[0]}  '
       f'logFC={top1_icd["logFC"].values[0]:.3f}  '
       f'FDR={top1_icd["adj_p_value"].values[0]:.2e}')

# IBDome UC
de_iuc = pd.read_csv(
    f'{TABLES_DIR}/ibdome_uc_de_v2.csv')
sig_iuc  = de_iuc[de_iuc['adj_p_value']<0.05]
up_h_iuc = sig_iuc[sig_iuc['logFC']>0]
top1_iuc = sig_iuc.nsmallest(
    1,'adj_p_value')
print(f'IBDome UC: sig={len(sig_iuc)}  '
       f'up_hyper={len(up_h_iuc)}  '
       f'up_quiet={len(sig_iuc)-len(up_h_iuc)}')
print(f'  Top: {top1_iuc["protein"].values[0]}  '
       f'logFC={top1_iuc["logFC"].values[0]:.3f}  '
       f'FDR={top1_iuc["adj_p_value"].values[0]:.2e}')

# ═══════════════════════════════════════
# 6. RF AUC
# ═══════════════════════════════════════
print('\n6. RANDOM FOREST AUC')
print('-'*40)

# Known from pipeline
rf_results = {
    'UKB CD'   : (0.983, 0.014),
    'UKB UC'   : (0.985, 0.003),
    'IBDome CD': (0.992, 0.006),
    'IBDome UC': (0.998, 0.003),
}
for name, (auc, std) in \
        rf_results.items():
    print(f'{name}:  '
           f'AUC={auc:.3f} '
           f'\u00b1{std:.3f}')

# ═══════════════════════════════════════
# 7. ORA TOP PATHWAYS
# ═══════════════════════════════════════
print('\n7. ORA TOP PATHWAYS')
print('-'*40)

import re
def clean_term(t):
    t = re.sub(r'HALLMARK_','',str(t))
    return t.replace('_',' ').strip()

for name, f in [
    ('UKB CD',
     'ora_cd_C1_up_Hallmark.csv'),
    ('UKB UC',
     'ora_uc_C1_up_Hallmark.csv'),
    ('IBDome CD',
     'ibdome_cd_ora_C2_up_Hallmark.csv'),
    ('IBDome UC',
     'ibdome_uc_ora_C2_up_Hallmark.csv'),
]:
    ora = pd.read_csv(
        f'{TABLES_DIR}/{f}')
    sig = ora[
        ora['Adjusted P-value']<0.05]
    top3 = sig.nsmallest(
        3,'Adjusted P-value')
    print(f'\n{name}: '
           f'{len(sig)} significant')
    for _, row in top3.iterrows():
        print(f'  {clean_term(row["Term"])[:35]}  '
               f'FDR={row["Adjusted P-value"]:.2e}')

# ═══════════════════════════════════════
# 8. VALIDATION METRICS
# ═══════════════════════════════════════
print('\n8. IBDOME VALIDATION')
print('-'*40)

import json
val_cd = json.load(open(
    f'{DATA_DIR}/'
    'ibdome_cd_validation_v2.json'))
val_uc = json.load(open(
    f'{DATA_DIR}/'
    'ibdome_uc_validation_v2.json'))

print('IBDome CD:')
for k,v in val_cd.items():
    if not isinstance(v, list):
        print(f'  {k}: {v}')

print('IBDome UC:')
for k,v in val_uc.items():
    if not isinstance(v, list):
        print(f'  {k}: {v}')

# ═══════════════════════════════════════
# 9. GMM ARI
# ═══════════════════════════════════════
print('\n9. GMM ARI')
print('-'*40)

from sklearn.metrics import (
    adjusted_rand_score)
from sklearn.mixture import (
    GaussianMixture)

for name, Z_f, lbl, hl in [
    ('UKB CD',
     'latent_cd_dae_dec.npy',
     lbl_cd, 0),
    ('UKB UC',
     'latent_uc_dae_dec.npy',
     lbl_uc, 0),
    ('IBDome CD',
     'ibdome_latent_dae_dec.npy',
     lbl_icd, 1),
    ('IBDome UC',
     'ibdome_uc_latent_dae_dec.npy',
     lbl_iuc, 1),
]:
    Z = np.load(
        f'{DATA_DIR}/{Z_f}')
    gmm = GaussianMixture(
        n_components=2,
        random_state=42,
        covariance_type='full')
    gmm_lbl = gmm.fit_predict(Z)
    ari = adjusted_rand_score(
        lbl, gmm_lbl)
    print(f'{name}:  ARI={ari:.3f}')

# ═══════════════════════════════════════
# 10. MISSING DATA
# ═══════════════════════════════════════
print('\n10. MISSING DATA CHECK')
print('-'*40)

print(f'X_cd NaN: {np.isnan(X_cd).sum()}')
print(f'X_uc NaN: {np.isnan(X_uc).sum()}')
print(f'X_icd NaN: {np.isnan(X_icd).sum()}')
print(f'X_iuc NaN: {np.isnan(X_iuc).sum()}')

print('\n' + '='*60)
print('VERIFICATION COMPLETE')
print('='*60)

In [ ]:
import numpy as np
from sklearn.metrics import adjusted_rand_score

DATA_DIR = '/rds/homes/j/jxt554/data'

lbl_cd = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')
lbl_uc = np.load(
    f'{DATA_DIR}/labels_uc_consensus.npy')

# Check saved GMM labels
for f, lbl, name in [
    ('labels_cd_gmm_best.npy',
     lbl_cd, 'UKB CD'),
    ('labels_uc_gmm_best.npy',
     lbl_uc, 'UKB UC'),
    ('labels_cd_gmm.npy',
     lbl_cd, 'UKB CD v2'),
    ('labels_uc_gmm.npy',
     lbl_uc, 'UKB UC v2'),
]:
    try:
        gmm = np.load(
            f'{DATA_DIR}/{f}')
        ari = adjusted_rand_score(
            lbl, gmm)
        print(f'{name} {f}: '
               f'ARI={ari:.3f}')
    except Exception as e:
        print(f'{f}: {e}')

In [ ]:
# Run this first to get
# the actual values

import pandas as pd

TABLES_DIR = '/rds/homes/j/jxt554/tables'

# Check exact columns
for f in [
    'cd_limma_M3.csv',
    'uc_limma_M3.csv',
    'ibdome_cd_de_v2.csv',
    'ibdome_uc_de_v2.csv']:
    df = pd.read_csv(
        f'{TABLES_DIR}/{f}')
    print(f'\n{f}: {df.shape}')
    print(df.columns.tolist())
    print(df.head(3))

In [ ]:
import pandas as pd
import numpy as np

TABLES_DIR = '/rds/homes/j/jxt554/tables'
OUTPUT     = '/rds/homes/j/jxt554/long_tables.tex'

# ── Load data ─────────────────────────
de_cd  = pd.read_csv(
    f'{TABLES_DIR}/cd_limma_M3.csv')
de_uc  = pd.read_csv(
    f'{TABLES_DIR}/uc_limma_M3.csv')
de_icd = pd.read_csv(
    f'{TABLES_DIR}/ibdome_cd_de_v2.csv')
de_iuc = pd.read_csv(
    f'{TABLES_DIR}/ibdome_uc_de_v2.csv')

# Fix protein col
for df in [de_cd, de_uc]:
    if 'protein' not in df.columns:
        df.rename(
            columns={df.columns[0]:
                      'protein'},
            inplace=True)

# Filter significant only
sig_cd  = de_cd[
    de_cd['adj.P.Val'] < 0.05
].sort_values('adj.P.Val').copy()
sig_uc  = de_uc[
    de_uc['adj.P.Val'] < 0.05
].sort_values('adj.P.Val').copy()
sig_icd = de_icd[
    de_icd['adj_p_value'] < 0.05
].sort_values('adj_p_value').copy()
sig_iuc = de_iuc[
    de_iuc['adj_p_value'] < 0.05
].sort_values('adj_p_value').copy()

print(f'UKB CD sig:    {len(sig_cd)}')
print(f'UKB UC sig:    {len(sig_uc)}')
print(f'IBDome CD sig: {len(sig_icd)}')
print(f'IBDome UC sig: {len(sig_iuc)}')

# ── Format helpers ────────────────────
def fmt_fc(v):
    return f'${v:+.3f}$'

def fmt_fdr(v):
    if v < 0.001:
        exp = int(np.floor(np.log10(v)))
        man = v / 10**exp
        return f'${man:.2f}\\!\\times\\!10^{{{exp}}}$'
    return f'${v:.4f}$'

def fmt_t(v):
    return f'${v:.3f}$'

def fmt_b(v):
    return f'${v:.2f}$'

def row_colour(i):
    return '\\rowcolor{tablerow}\n' \
        if i % 2 == 1 else ''

# ── UKB header colour ─────────────────
HEADER = '\\rowcolor{tableheader}'
HC     = '\\textcolor{white}'

# ── Build LaTeX ───────────────────────
out = []

out.append(
    '% ═══════════════════════════\n'
    '% LONG TABLES\n'
    '% ═══════════════════════════\n'
    '\\usepackage{longtable}\n'
    '\\usepackage{booktabs}\n\n')

# ════════════════════════════════
# TABLE S12 — UKB CD full DE
# ════════════════════════════════
out.append(r"""
\clearpage
\begin{longtable}{lrrrrr}
\caption[Full DE results UKB CD]{
    \textbf{Supplementary Table S12
    --- Full Differential Protein
    Abundance Results:
    UK Biobank Crohn's Disease.}
    All 615 proteins significant at
    FDR\,$<$\,0.05 from \textit{limma}
    empirical Bayes Model 3
    (subtype\,$+$\,batch\,$+$\,stability).
    Negative logFC indicates
    elevation in the
    Hyperinflammatory subtype
    (cluster 0).
    Sorted by FDR ascending.
    logFC: log$_2$ fold change.
    AveExpr: average expression.
    $t$: moderated $t$-statistic.
    FDR: Benjamini-Hochberg
    adjusted $p$-value.
    $B$: log-odds of DE.}
\label{tab:s12_ukb_cd} \\
""")

out.append(
    HEADER + '\n'
    + HC + r'{\textbf{Protein}}'
    + ' & '
    + HC + r'{\textbf{logFC}}'
    + ' & '
    + HC + r'{\textbf{AveExpr}}'
    + ' & '
    + HC + r'{\textbf{$t$}}'
    + ' & '
    + HC + r'{\textbf{FDR}}'
    + ' & '
    + HC + r'{\textbf{$B$}}'
    + r' \\' + '\n'
    + r'\toprule' + '\n'
    + r'\endfirsthead' + '\n\n'
    + HEADER + '\n'
    + HC + r'{\textbf{Protein}}'
    + ' & '
    + HC + r'{\textbf{logFC}}'
    + ' & '
    + HC + r'{\textbf{AveExpr}}'
    + ' & '
    + HC + r'{\textbf{$t$}}'
    + ' & '
    + HC + r'{\textbf{FDR}}'
    + ' & '
    + HC + r'{\textbf{$B$}}'
    + r' \\' + '\n'
    + r'\toprule' + '\n'
    + r'\endhead' + '\n\n'
    + r'\midrule' + '\n'
    + r'\multicolumn{6}{r}{'
    + r'\footnotesize Continued '
    + r'on next page} \\'
    + '\n'
    + r'\endfoot' + '\n\n'
    + r'\bottomrule' + '\n'
    + r'\multicolumn{6}{l}{'
    + r'\footnotesize '
    + r'logFC: negative = '
    + r'elevated in '
    + r'Hyperinflammatory. '
    + r'FDR: BH corrected.}'
    + r' \\' + '\n'
    + r'\endlastfoot' + '\n\n')

for i, (_, row) in enumerate(
        sig_cd.iterrows()):
    out.append(
        row_colour(i) +
        f'{row["protein"]} & '
        f'{fmt_fc(row["logFC"])} & '
        f'{row["AveExpr"]:.3f} & '
        f'{fmt_t(row["t"])} & '
        f'{fmt_fdr(row["adj.P.Val"])} & '
        f'{fmt_b(row["B"])} \\\\\n')

out.append('\\end{longtable}\n\n')

# ════════════════════════════════
# TABLE S13 — UKB UC full DE
# ════════════════════════════════
out.append(r"""
\clearpage
\begin{longtable}{lrrrrr}
\caption[Full DE results UKB UC]{
    \textbf{Supplementary Table S13
    --- Full Differential Protein
    Abundance Results:
    UK Biobank Ulcerative Colitis.}
    All 647 proteins significant at
    FDR\,$<$\,0.05 from \textit{limma}
    empirical Bayes Model 3.
    Negative logFC indicates
    elevation in Hyperinflammatory.
    Sorted by FDR ascending.}
\label{tab:s13_ukb_uc} \\
""")

out.append(
    HEADER + '\n'
    + HC + r'{\textbf{Protein}}'
    + ' & '
    + HC + r'{\textbf{logFC}}'
    + ' & '
    + HC + r'{\textbf{AveExpr}}'
    + ' & '
    + HC + r'{\textbf{$t$}}'
    + ' & '
    + HC + r'{\textbf{FDR}}'
    + ' & '
    + HC + r'{\textbf{$B$}}'
    + r' \\' + '\n'
    + r'\toprule' + '\n'
    + r'\endfirsthead' + '\n\n'
    + HEADER + '\n'
    + HC + r'{\textbf{Protein}}'
    + ' & '
    + HC + r'{\textbf{logFC}}'
    + ' & '
    + HC + r'{\textbf{AveExpr}}'
    + ' & '
    + HC + r'{\textbf{$t$}}'
    + ' & '
    + HC + r'{\textbf{FDR}}'
    + ' & '
    + HC + r'{\textbf{$B$}}'
    + r' \\' + '\n'
    + r'\toprule' + '\n'
    + r'\endhead' + '\n\n'
    + r'\midrule' + '\n'
    + r'\multicolumn{6}{r}{'
    + r'\footnotesize Continued '
    + r'on next page} \\'
    + '\n'
    + r'\endfoot' + '\n\n'
    + r'\bottomrule' + '\n'
    + r'\multicolumn{6}{l}{'
    + r'\footnotesize '
    + r'logFC: negative = '
    + r'elevated in '
    + r'Hyperinflammatory. '
    + r'FDR: BH corrected.}'
    + r' \\' + '\n'
    + r'\endlastfoot' + '\n\n')

for i, (_, row) in enumerate(
        sig_uc.iterrows()):
    out.append(
        row_colour(i) +
        f'{row["protein"]} & '
        f'{fmt_fc(row["logFC"])} & '
        f'{row["AveExpr"]:.3f} & '
        f'{fmt_t(row["t"])} & '
        f'{fmt_fdr(row["adj.P.Val"])} & '
        f'{fmt_b(row["B"])} \\\\\n')

out.append('\\end{longtable}\n\n')

# ════════════════════════════════
# TABLE S14 — IBDome CD full DE
# ════════════════════════════════
out.append(r"""
\clearpage
\begin{longtable}{lrrrrrr}
\caption[Full DE results IBDome CD]{
    \textbf{Supplementary Table S14
    --- Full Differential Protein
    Abundance Results:
    IBDome Crohn's Disease.}
    All 58 of 61 proteins
    significant at FDR\,$<$\,0.05
    from Welch $t$-test.
    Positive logFC indicates
    elevation in Hyperinflammatory
    (cluster 1).
    mean\_H: mean expression in
    Hyperinflammatory;
    mean\_Q: mean expression in
    Quiescent.
    Sorted by FDR ascending.}
\label{tab:s14_ibd_cd} \\
""")

out.append(
    HEADER + '\n'
    + HC + r'{\textbf{Protein}}'
    + ' & '
    + HC + r'{\textbf{logFC}}'
    + ' & '
    + HC + r'{\textbf{$t$}}'
    + ' & '
    + HC + r'{\textbf{$p$-value}}'
    + ' & '
    + HC + r'{\textbf{FDR}}'
    + ' & '
    + HC + r'{\textbf{mean\_H}}'
    + ' & '
    + HC + r'{\textbf{mean\_Q}}'
    + r' \\' + '\n'
    + r'\toprule' + '\n'
    + r'\endfirsthead' + '\n\n'
    + HEADER + '\n'
    + HC + r'{\textbf{Protein}}'
    + ' & '
    + HC + r'{\textbf{logFC}}'
    + ' & '
    + HC + r'{\textbf{$t$}}'
    + ' & '
    + HC + r'{\textbf{$p$-value}}'
    + ' & '
    + HC + r'{\textbf{FDR}}'
    + ' & '
    + HC + r'{\textbf{mean\_H}}'
    + ' & '
    + HC + r'{\textbf{mean\_Q}}'
    + r' \\' + '\n'
    + r'\toprule' + '\n'
    + r'\endhead' + '\n\n'
    + r'\midrule' + '\n'
    + r'\multicolumn{7}{r}{'
    + r'\footnotesize Continued '
    + r'on next page} \\'
    + '\n'
    + r'\endfoot' + '\n\n'
    + r'\bottomrule' + '\n'
    + r'\multicolumn{7}{l}{'
    + r'\footnotesize '
    + r'logFC: positive = '
    + r'elevated in '
    + r'Hyperinflammatory. '
    + r'FDR: BH corrected. '
    + r'Welch $t$-test.}'
    + r' \\' + '\n'
    + r'\endlastfoot' + '\n\n')

for i, (_, row) in enumerate(
        sig_icd.iterrows()):
    out.append(
        row_colour(i) +
        f'{row["protein"]} & '
        f'{fmt_fc(row["logFC"])} & '
        f'{fmt_t(row["t_stat"])} & '
        f'{fmt_fdr(row["p_value"])} & '
        f'{fmt_fdr(row["adj_p_value"])} & '
        f'{row["mean_C2"]:.3f} & '
        f'{row["mean_C1"]:.3f} \\\\\n')

out.append('\\end{longtable}\n\n')

# ════════════════════════════════
# TABLE S15 — IBDome UC full DE
# ════════════════════════════════
out.append(r"""
\clearpage
\begin{longtable}{lrrrrrr}
\caption[Full DE results IBDome UC]{
    \textbf{Supplementary Table S15
    --- Full Differential Protein
    Abundance Results:
    IBDome Ulcerative Colitis.}
    All 53 of 61 proteins
    significant at FDR\,$<$\,0.05
    from Welch $t$-test.
    Positive logFC indicates
    elevation in Hyperinflammatory
    (cluster 1).
    Sorted by FDR ascending.}
\label{tab:s15_ibd_uc} \\
""")

out.append(
    HEADER + '\n'
    + HC + r'{\textbf{Protein}}'
    + ' & '
    + HC + r'{\textbf{logFC}}'
    + ' & '
    + HC + r'{\textbf{$t$}}'
    + ' & '
    + HC + r'{\textbf{$p$-value}}'
    + ' & '
    + HC + r'{\textbf{FDR}}'
    + ' & '
    + HC + r'{\textbf{mean\_H}}'
    + ' & '
    + HC + r'{\textbf{mean\_Q}}'
    + r' \\' + '\n'
    + r'\toprule' + '\n'
    + r'\endfirsthead' + '\n\n'
    + HEADER + '\n'
    + HC + r'{\textbf{Protein}}'
    + ' & '
    + HC + r'{\textbf{logFC}}'
    + ' & '
    + HC + r'{\textbf{$t$}}'
    + ' & '
    + HC + r'{\textbf{$p$-value}}'
    + ' & '
    + HC + r'{\textbf{FDR}}'
    + ' & '
    + HC + r'{\textbf{mean\_H}}'
    + ' & '
    + HC + r'{\textbf{mean\_Q}}'
    + r' \\' + '\n'
    + r'\toprule' + '\n'
    + r'\endhead' + '\n\n'
    + r'\midrule' + '\n'
    + r'\multicolumn{7}{r}{'
    + r'\footnotesize Continued '
    + r'on next page} \\'
    + '\n'
    + r'\endfoot' + '\n\n'
    + r'\bottomrule' + '\n'
    + r'\multicolumn{7}{l}{'
    + r'\footnotesize '
    + r'logFC: positive = '
    + r'elevated in '
    + r'Hyperinflammatory. '
    + r'FDR: BH corrected. '
    + r'Welch $t$-test.}'
    + r' \\' + '\n'
    + r'\endlastfoot' + '\n\n')

for i, (_, row) in enumerate(
        sig_iuc.iterrows()):
    out.append(
        row_colour(i) +
        f'{row["protein"]} & '
        f'{fmt_fc(row["logFC"])} & '
        f'{fmt_t(row["t_stat"])} & '
        f'{fmt_fdr(row["p_value"])} & '
        f'{fmt_fdr(row["adj_p_value"])} & '
        f'{row["mean_C2"]:.3f} & '
        f'{row["mean_C1"]:.3f} \\\\\n')

out.append('\\end{longtable}\n\n')

# ── Write file ────────────────────────
with open(OUTPUT, 'w') as f:
    f.write(''.join(out))

print(f'Written to {OUTPUT}')
print(f'Total lines: '
       f'{len("".join(out).splitlines())}')
print('\nRow counts:')
print(f'  S12 UKB CD:    {len(sig_cd)} rows')
print(f'  S13 UKB UC:    {len(sig_uc)} rows')
print(f'  S14 IBDome CD: {len(sig_icd)} rows')
print(f'  S15 IBDome UC: {len(sig_iuc)} rows')

In [ ]:
# Just read and print the content
# so you can copy paste it

with open(
    '/rds/homes/j/jxt554/long_tables.tex',
    'r') as f:
    content = f.read()

print(f'File size: '
       f'{len(content)/1024:.1f} KB')
print(f'Lines: '
       f'{len(content.splitlines())}')
print('\nFirst 50 lines:')
for line in content.splitlines()[:50]:
    print(line)

In [ ]:
import shutil

shutil.copy(
    '/rds/homes/j/jxt554/long_tables.tex',
    '/rds/homes/j/jxt554/figures/long_tables.tex')

print('Copied to figures folder')
print('Download from there')